[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/bike-availability-data-science-full/blob/main/notebooks/Module_02_Data_Acquisition/M2_03_data_storage.ipynb)

# 💾 Module 02: Data Storage Patterns - SOLUTIONS

**Purpose**: Complete solutions for data storage best practices  
**Module**: Module 02 - Data Acquisition  
**Author**: Course Team  
**Date**: 2026-01-16

---

## 📋 Overview

In this notebook, you will:
- [ ] Compare file formats (CSV, JSON, Parquet)
- [ ] Understand storage trade-offs (size, speed, compatibility)
- [ ] Implement data versioning strategies
- [ ] Set up proper `.gitignore` for data files
- [ ] Document data lineage and transformations
- [ ] Create efficient data storage pipelines

**Goal**: Establish professional data management practices!

**Estimated Time**: 60-90 minutes

---

## 📖 Part 1: Why Storage Matters

### The Data Storage Challenge

In data science projects, you'll work with:
- 📥 **Raw data** - Original, unmodified datasets
- 🔧 **Intermediate data** - Partially processed results
- 📊 **Final data** - Analysis-ready datasets
- 💾 **Model artifacts** - Trained models, predictions

**Poor storage practices lead to**:
- ❌ Lost data (no backups)
- ❌ Version confusion (which is latest?)
- ❌ Wasted space (inefficient formats)
- ❌ Slow processing (wrong format choice)
- ❌ Collaboration issues (inconsistent organization)

### Storage Principles

1. **🔒 Immutability** - Never modify raw data
2. **📝 Documentation** - Always document transformations
3. **🔢 Versioning** - Track changes over time
4. **⚡ Efficiency** - Choose appropriate formats
5. **🤝 Accessibility** - Make data discoverable

---

## 📊 File Format Comparison

| Format | Best For | Pros | Cons | Size | Speed |
|--------|----------|------|------|------|-------|
| **CSV** | Simple tabular data | Human-readable, universal | Large files, no types | ⭐⭐ | ⭐⭐ |
| **JSON** | Nested/hierarchical data | Flexible structure | Verbose, slow parsing | ⭐ | ⭐ |
| **Parquet** | Large datasets, analytics | Compressed, typed, fast | Binary (not readable) | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Feather** | Intermediate processing | Very fast I/O | Less compression | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **HDF5** | Multi-dimensional arrays | Efficient for NumPy | Complex, not portable | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |

**Rule of thumb**:
- 📁 Raw data → Keep original format (JSON, CSV)
- 🔧 Processed data → Parquet (best all-around)
- ⚡ Temporary data → Feather (fastest I/O)

---

## 🔧 Part 2: Setup

Run this cell first to set up the environment.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════

# Standard library
import os
import sys
import json
import time
from datetime import datetime
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    print("📍 Running in Google Colab")
else:
    print("📍 Running locally")

# Add project root to path
project_root = os.path.abspath('../..' if 'notebooks' in os.getcwd() else '.')
if project_root not in sys.path:
    sys.path.append(project_root)

print("✅ Setup complete!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Part 3: Create Sample Dataset

Let's create a sample bike availability dataset to experiment with.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. CREATE SAMPLE DATA
# ═══════════════════════════════════════════════════════════

# Create a realistic sample dataset
np.random.seed(42)

# Generate 30 days of hourly data (720 records)
hours = pd.date_range('2026-01-01', periods=720, freq='H')

# Simulate 10 bike stations
stations = [f'Station_{i:03d}' for i in range(1, 11)]

# Create data for each station
data_list = []
for station in stations:
    for timestamp in hours:
        # Add some realistic patterns
        hour = timestamp.hour
        day_of_week = timestamp.dayofweek
        
        # More bikes available during commute hours
        base_bikes = 10
        if hour in [8, 9, 17, 18]:
            base_bikes += np.random.randint(5, 15)
        
        # Fewer bikes on weekends
        if day_of_week >= 5:
            base_bikes = int(base_bikes * 0.7)
        
        data_list.append({
            'timestamp': timestamp,
            'station_id': station,
            'station_name': f'Amsterdam {station}',
            'bikes_available': base_bikes + np.random.randint(-5, 5),
            'docks_available': 20 - (base_bikes + np.random.randint(-5, 5)),
            'latitude': 52.37 + np.random.uniform(-0.05, 0.05),
            'longitude': 4.90 + np.random.uniform(-0.05, 0.05),
            'temperature_c': 15 + np.random.uniform(-10, 10),
            'precipitation_mm': max(0, np.random.exponential(0.5)),
            'wind_speed_kmh': max(0, np.random.normal(15, 5))
        })

df_sample = pd.DataFrame(data_list)

# Clean up negative values
df_sample['bikes_available'] = df_sample['bikes_available'].clip(lower=0)
df_sample['docks_available'] = df_sample['docks_available'].clip(lower=0)

print(f"✅ Created sample dataset with {len(df_sample):,} rows")
print(f"📊 Shape: {df_sample.shape}")
print(f"📅 Date range: {df_sample['timestamp'].min()} to {df_sample['timestamp'].max()}")
print(f"\n📋 First few rows:")
display(df_sample.head())

---

## 📁 Part 4: File Format Experiments

Let's compare different file formats by saving and loading the same data.

### Setup Test Directory

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. SETUP TEST DIRECTORY
# ═══════════════════════════════════════════════════════════

# Create temporary directory for experiments
TEST_DIR = Path('../../data/processed') if 'notebooks' in os.getcwd() else Path('data/processed')
TEST_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Test directory: {TEST_DIR}")
print(f"✅ Directory ready for format experiments")

### Experiment 1: Save in Different Formats

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. FORMAT COMPARISON - SAVE
# ═══════════════════════════════════════════════════════════

formats = {}

# 1. CSV Format
print("💾 Testing CSV format...")
csv_file = TEST_DIR / 'bike_data_test.csv'
start_time = time.time()
df_sample.to_csv(csv_file, index=False)
csv_time = time.time() - start_time
csv_size = csv_file.stat().st_size

formats['CSV'] = {
    'file': csv_file,
    'save_time': csv_time,
    'size_bytes': csv_size,
    'size_mb': csv_size / 1024 / 1024
}
print(f"  ✅ Saved in {csv_time:.3f}s, Size: {csv_size/1024:.1f} KB")

# 2. JSON Format
print("\n💾 Testing JSON format...")
json_file = TEST_DIR / 'bike_data_test.json'
start_time = time.time()
df_sample.to_json(json_file, orient='records', date_format='iso', indent=2)
json_time = time.time() - start_time
json_size = json_file.stat().st_size

formats['JSON'] = {
    'file': json_file,
    'save_time': json_time,
    'size_bytes': json_size,
    'size_mb': json_size / 1024 / 1024
}
print(f"  ✅ Saved in {json_time:.3f}s, Size: {json_size/1024:.1f} KB")

# 3. Parquet Format
print("\n💾 Testing Parquet format...")
parquet_file = TEST_DIR / 'bike_data_test.parquet'
start_time = time.time()
df_sample.to_parquet(parquet_file, index=False, compression='snappy')
parquet_time = time.time() - start_time
parquet_size = parquet_file.stat().st_size

formats['Parquet'] = {
    'file': parquet_file,
    'save_time': parquet_time,
    'size_bytes': parquet_size,
    'size_mb': parquet_size / 1024 / 1024
}
print(f"  ✅ Saved in {parquet_time:.3f}s, Size: {parquet_size/1024:.1f} KB")

# 4. Feather Format (if available)
try:
    print("\n💾 Testing Feather format...")
    feather_file = TEST_DIR / 'bike_data_test.feather'
    start_time = time.time()
    df_sample.to_feather(feather_file)
    feather_time = time.time() - start_time
    feather_size = feather_file.stat().st_size
    
    formats['Feather'] = {
        'file': feather_file,
        'save_time': feather_time,
        'size_bytes': feather_size,
        'size_mb': feather_size / 1024 / 1024
    }
    print(f"  ✅ Saved in {feather_time:.3f}s, Size: {feather_size/1024:.1f} KB")
except Exception as e:
    print(f"  ⚠️ Feather not available: {e}")

print("\n" + "=" * 60)
print("📊 Save Performance Summary:")
print("=" * 60)
for fmt, info in formats.items():
    print(f"{fmt:10s}: {info['save_time']:6.3f}s, {info['size_mb']:8.2f} MB")

### Experiment 2: Load from Different Formats

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. FORMAT COMPARISON - LOAD
# ═══════════════════════════════════════════════════════════

print("📂 Testing load performance...")
print("=" * 60)

# Test each format
for fmt, info in formats.items():
    file_path = info['file']
    
    start_time = time.time()
    
    if fmt == 'CSV':
        df_loaded = pd.read_csv(file_path, parse_dates=['timestamp'])
    elif fmt == 'JSON':
        df_loaded = pd.read_json(file_path)
    elif fmt == 'Parquet':
        df_loaded = pd.read_parquet(file_path)
    elif fmt == 'Feather':
        df_loaded = pd.read_feather(file_path)
    
    load_time = time.time() - start_time
    formats[fmt]['load_time'] = load_time
    
    print(f"{fmt:10s}: Loaded {len(df_loaded):,} rows in {load_time:.3f}s")

print("\n" + "=" * 60)
print("📊 Complete Performance Summary:")
print("=" * 60)
print(f"{'Format':<10} {'Save (s)':<10} {'Load (s)':<10} {'Size (MB)':<12} {'Compression':<12}")
print("-" * 60)

# Calculate compression ratios relative to CSV
csv_size = formats['CSV']['size_mb']
for fmt, info in formats.items():
    compression_ratio = (1 - info['size_mb'] / csv_size) * 100
    print(f"{fmt:<10} {info['save_time']:>8.3f}  {info['load_time']:>8.3f}  "
          f"{info['size_mb']:>10.2f}  {compression_ratio:>10.1f}%")

### Visualization: Format Comparison

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. VISUALIZE FORMAT COMPARISON
# ═══════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('📊 File Format Comparison', fontsize=16, fontweight='bold')

format_names = list(formats.keys())
save_times = [formats[f]['save_time'] for f in format_names]
load_times = [formats[f]['load_time'] for f in format_names]
sizes_mb = [formats[f]['size_mb'] for f in format_names]

# 1. Save time comparison
axes[0].bar(format_names, save_times, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Save Performance\n(Lower is Better)')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(save_times):
    axes[0].text(i, v, f'{v:.3f}s', ha='center', va='bottom', fontweight='bold')

# 2. Load time comparison
axes[1].bar(format_names, load_times, color='darkorange', alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Load Performance\n(Lower is Better)')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(load_times):
    axes[1].text(i, v, f'{v:.3f}s', ha='center', va='bottom', fontweight='bold')

# 3. File size comparison
axes[2].bar(format_names, sizes_mb, color='green', alpha=0.7, edgecolor='black')
axes[2].set_ylabel('File Size (MB)')
axes[2].set_title('Storage Size\n(Lower is Better)')
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(sizes_mb):
    axes[2].text(i, v, f'{v:.2f}MB', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Recommendations
print("\n" + "=" * 60)
print("💡 Format Recommendations:")
print("=" * 60)
print("📁 Raw Data (preserve original):")
print("   → CSV/JSON - Keep in original format for traceability")
print("\n📊 Processed Data (analysis-ready):")
print("   → Parquet - Best balance of size, speed, and features")
print("\n⚡ Temporary/Intermediate Data:")
print("   → Feather - Fastest I/O for temporary workflows")
print("\n🤝 Sharing with Non-Technical Users:")
print("   → CSV - Universal compatibility, human-readable")

---

### 🧠 Task 4.1: Visualize Format Comparison (70% Scaffolding)

**Your Task**: Create visualizations comparing the file format performance.

**What you'll learn**: 
- How to interpret performance metrics
- Making data-driven format choices
- Creating comparison visualizations

**Requirements**:
1. Create a bar chart comparing file sizes across formats
2. Create a bar chart comparing save times across formats  
3. Add a recommendation based on your findings

**Hints**:
- Use the `formats` dictionary created above
- Try `plt.subplot(1, 2, 1)` for side-by-side plots
- Consider: Which format has the best size/speed trade-off?

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.1 SOLUTION: FORMAT VISUALIZATION
# ═══════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

# Create figure with 2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📊 File Format Comparison - Solution', fontsize=14, fontweight='bold')

# Extract data from formats dictionary
format_names = list(formats.keys())
sizes_mb = [formats[fmt]['size_mb'] for fmt in format_names]
save_times = [formats[fmt]['save_time'] for fmt in format_names]

# Plot 1: File sizes
ax1.bar(format_names, sizes_mb, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], 
        alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('File Size (MB)', fontsize=11, fontweight='bold')
ax1.set_title('Storage Size Comparison\n(Lower is Better)', fontsize=12)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_axisbelow(True)

# Add value labels on bars
for i, v in enumerate(sizes_mb):
    ax1.text(i, v + 0.05, f'{v:.2f}MB', ha='center', va='bottom', 
             fontweight='bold', fontsize=10)

# Plot 2: Save times
ax2.bar(format_names, save_times, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'],
        alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Save Time (seconds)', fontsize=11, fontweight='bold')
ax2.set_title('Write Performance\n(Lower is Better)', fontsize=12)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.set_axisbelow(True)

# Add value labels
for i, v in enumerate(save_times):
    ax2.text(i, v + 0.001, f'{v:.3f}s', ha='center', va='bottom',
             fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# Analysis and recommendations
print("\n" + "="*70)
print("📊 SOLUTION: Format Recommendations")
print("="*70)

# Find best for each metric
best_size = min(format_names, key=lambda f: formats[f]['size_mb'])
best_speed = min(format_names, key=lambda f: formats[f]['save_time'])

# Calculate balance score (normalized size + speed)
scores = {}
for fmt in format_names:
    norm_size = formats[fmt]['size_mb'] / max(sizes_mb)
    norm_speed = formats[fmt]['save_time'] / max(save_times)
    scores[fmt] = norm_size + norm_speed

best_balance = min(scores, key=scores.get)

print(f"\nFor this dataset size ({len(df_sample):,} rows):")
print(f"  🏆 Best for SIZE:  {best_size} ({formats[best_size]['size_mb']:.2f} MB)")
print(f"  🏆 Best for SPEED: {best_speed} ({formats[best_speed]['save_time']:.3f}s)")
print(f"  🏆 Best BALANCE:   {best_balance} (size + speed optimized)")

print("\n💡 Real-World Guidance:")
print("  • Parquet: Best all-around - use for processed data storage")
print("  • Feather: Fastest I/O - use for temporary/intermediate files")
print("  • CSV: Human-readable - use for raw data and sharing")
print("  • JSON: Hierarchical - use when structure matters over performance")

print("\n🎓 Learning Points:")
print("  ✓ Parquet offers 60-80% compression vs CSV")
print("  ✓ Binary formats (Parquet/Feather) are 5-10x faster to read")
print("  ✓ Choice depends on: data size, update frequency, and audience")

---

### 🧠 Task 4.2: Performance Testing with Larger Data (60% Scaffolding)

**Your Task**: Test read performance with a 10x larger dataset.

**What you'll learn**:
- How dataset size affects format choice
- Statistical performance testing
- When to use each format in production

**Requirements**:
1. Create a dataset with 7,200 rows (10x current size)
2. Save in all 4 formats
3. Test read performance 5 times each
4. Calculate mean and standard deviation
5. Generate performance report

**Provided structure** - Complete the TODOs:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 4.2 SOLUTION: PERFORMANCE TESTING
# ═══════════════════════════════════════════════════════════

import time
import numpy as np

print("🔬 Creating larger dataset for performance testing...")

# Create 10x larger dataset (7,200 rows)
df_large = pd.concat([df_sample] * 10, ignore_index=True)
print(f"✅ Created dataset with {len(df_large):,} rows")
print(f"   Original: {len(df_sample):,} rows")
print(f"   Multiplier: 10x\n")

# Save in all formats
print("💾 Saving large dataset in all formats...")
large_files = {}

for fmt_name in format_names:
    file_path = TEST_DIR / f'large_test.{fmt_name.lower()}'
    
    start = time.time()
    if fmt_name == 'CSV':
        df_large.to_csv(file_path, index=False)
    elif fmt_name == 'JSON':
        df_large.to_json(file_path, orient='records', date_format='iso')
    elif fmt_name == 'Parquet':
        df_large.to_parquet(file_path, index=False, compression='snappy')
    elif fmt_name == 'Feather':
        df_large.to_feather(file_path)
    
    save_time = time.time() - start
    file_size = file_path.stat().st_size / 1024 / 1024  # MB
    
    large_files[fmt_name] = {
        'path': file_path,
        'save_time': save_time,
        'size_mb': file_size
    }
    print(f"  {fmt_name}: {save_time:.3f}s, {file_size:.2f} MB")

# Run performance tests
print("\n⚡ Testing READ performance (5 trials each)...\n")
NUM_TRIALS = 5

results = {}
for fmt_name in format_names:
    read_times = []
    file_path = large_files[fmt_name]['path']
    
    for trial in range(NUM_TRIALS):
        start = time.time()
        
        if fmt_name == 'CSV':
            df_test = pd.read_csv(file_path, parse_dates=['timestamp'])
        elif fmt_name == 'JSON':
            df_test = pd.read_json(file_path)
        elif fmt_name == 'Parquet':
            df_test = pd.read_parquet(file_path)
        elif fmt_name == 'Feather':
            df_test = pd.read_feather(file_path)
        
        read_time = time.time() - start
        read_times.append(read_time)
    
    # Calculate statistics
    mean_time = np.mean(read_times)
    std_time = np.std(read_times)
    
    results[fmt_name] = {
        'times': read_times,
        'mean': mean_time,
        'std': std_time,
        'min': np.min(read_times),
        'max': np.max(read_times)
    }

# Display performance report
print("="*80)
print(f"📊 SOLUTION: Performance Report - Large Dataset ({len(df_large):,} rows)")
print("="*80)
print(f"{'Format':<12} {'Mean (s)':<12} {'Std Dev (s)':<14} {'Min (s)':<10} {'Max (s)':<10}")
print("-"*80)

for fmt_name in format_names:
    r = results[fmt_name]
    print(f"{fmt_name:<12} {r['mean']:>10.4f}  {r['std']:>12.5f}  "
          f"{r['min']:>8.4f}  {r['max']:>8.4f}")

print("-"*80)

# Performance analysis
fastest = min(results.items(), key=lambda x: x[1]['mean'])
slowest = max(results.items(), key=lambda x: x[1]['mean'])
speedup = slowest[1]['mean'] / fastest[1]['mean']

print("\n💡 Performance Insights:")
print(f"  🏆 Fastest: {fastest[0]} ({fastest[1]['mean']:.4f}s average)")
print(f"  🐌 Slowest: {slowest[0]} ({slowest[1]['mean']:.4f}s average)")
print(f"  ⚡ Speed-up: {speedup:.1f}x faster")

print("\n📈 Comparison to Small Dataset (720 rows):")
for fmt_name in format_names:
    if fmt_name in formats:
        small_time = formats[fmt_name].get('load_time', 0)
        large_time = results[fmt_name]['mean']
        if small_time > 0:
            ratio = large_time / small_time
            print(f"  {fmt_name}: {ratio:.1f}x slower (expected ~10x for linear scaling)")

print("\n🎓 Learning Points:")
print("  ✓ Binary formats (Parquet/Feather) scale better with size")
print("  ✓ JSON suffers most at larger sizes due to parsing overhead")
print("  ✓ Consistent performance (low std dev) is important for production")
print("  ✓ Always benchmark with realistic data sizes for your use case")

# Cleanup
for fmt_name in format_names:
    if large_files[fmt_name]['path'].exists():
        large_files[fmt_name]['path'].unlink()
print("\n🗑️  Temporary files cleaned up")

---

### 🧠 Task 4.2: Deep Dive - Read Performance (60% Scaffolding)

**Your Task**: Test read performance with larger datasets to see format differences magnified.

**What you'll learn**:
- How dataset size affects format choice
- Practical performance testing
- When to use each format

**Requirements**:
1. Create a 10x larger dataset (7,200 rows)
2. Save in all formats
3. Test read performance 5 times and calculate average
4. Create a performance report

**Provided code structure** - Fill in the TODOs:

---

## 🔢 Part 5: Data Versioning Strategies

### Versioning Approaches

1. **Timestamp-based**: `data_2026-01-15_143022.csv`
2. **Version numbers**: `data_v1.csv`, `data_v2.csv`
3. **Semantic versioning**: `data_v1.2.3.csv`
4. **Git LFS**: Version control for large files
5. **Data catalogs**: Metadata-driven (DVC, Pachyderm)

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. VERSIONING IMPLEMENTATION
# ═══════════════════════════════════════════════════════════

class DataVersionManager:
    """
    Simple data versioning system with metadata tracking.
    """
    
    def __init__(self, base_dir):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.metadata_file = self.base_dir / 'versions.json'
        self.metadata = self._load_metadata()
    
    def _load_metadata(self):
        """Load version metadata from file."""
        if self.metadata_file.exists():
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return {'versions': []}
    
    def _save_metadata(self):
        """Save version metadata to file."""
        with open(self.metadata_file, 'w') as f:
            json.dump(self.metadata, f, indent=2)
    
    def save_version(self, df, name, description='', version=None):
        """
        Save a new version of the dataset.
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Data to save
        name : str
            Base name for the dataset
        description : str
            Description of changes in this version
        version : int, optional
            Specific version number (auto-increment if None)
        
        Returns:
        --------
        dict : Version metadata
        """
        # Determine version number
        if version is None:
            existing_versions = [v['version'] for v in self.metadata['versions'] 
                                if v['name'] == name]
            version = max(existing_versions, default=0) + 1
        
        # Create filename
        timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
        filename = f"{name}_v{version}_{timestamp}.parquet"
        filepath = self.base_dir / filename
        
        # Save data
        df.to_parquet(filepath, index=False)
        
        # Create version metadata
        version_info = {
            'name': name,
            'version': version,
            'filename': filename,
            'timestamp': timestamp,
            'datetime': datetime.now().isoformat(),
            'description': description,
            'rows': len(df),
            'columns': len(df.columns),
            'size_bytes': filepath.stat().st_size,
            'column_names': list(df.columns)
        }
        
        # Update metadata
        self.metadata['versions'].append(version_info)
        self._save_metadata()
        
        print(f"✅ Saved version {version} of '{name}'")
        print(f"   File: {filename}")
        print(f"   Rows: {len(df):,}, Size: {filepath.stat().st_size/1024:.1f} KB")
        
        return version_info
    
    def load_version(self, name, version=None):
        """
        Load a specific version of the dataset.
        
        Parameters:
        -----------
        name : str
            Dataset name
        version : int, optional
            Version number (latest if None)
        
        Returns:
        --------
        pandas.DataFrame : Loaded data
        """
        # Find matching versions
        matching = [v for v in self.metadata['versions'] if v['name'] == name]
        
        if not matching:
            raise ValueError(f"No versions found for dataset '{name}'")
        
        # Get specific or latest version
        if version is None:
            version_info = max(matching, key=lambda x: x['version'])
        else:
            version_info = next((v for v in matching if v['version'] == version), None)
            if version_info is None:
                raise ValueError(f"Version {version} not found for '{name}'")
        
        # Load data
        filepath = self.base_dir / version_info['filename']
        df = pd.read_parquet(filepath)
        
        print(f"✅ Loaded version {version_info['version']} of '{name}'")
        print(f"   Date: {version_info['datetime']}")
        print(f"   Description: {version_info['description']}")
        print(f"   Rows: {len(df):,}")
        
        return df
    
    def list_versions(self, name=None):
        """List all versions or versions for specific dataset."""
        if name:
            versions = [v for v in self.metadata['versions'] if v['name'] == name]
        else:
            versions = self.metadata['versions']
        
        if not versions:
            print(f"No versions found{' for ' + name if name else ''}")
            return
        
        print(f"\n📚 Available Versions{' for ' + name if name else ''}:")
        print("=" * 80)
        print(f"{'Name':<20} {'Ver':<5} {'Date':<20} {'Rows':<10} {'Description':<30}")
        print("-" * 80)
        
        for v in sorted(versions, key=lambda x: (x['name'], x['version'])):
            print(f"{v['name']:<20} {v['version']:<5} {v['datetime'][:19]:<20} "
                  f"{v['rows']:<10,} {v['description']:<30}")


# Example usage
print("🔧 Setting up version manager...")
vm = DataVersionManager(TEST_DIR)

# Save first version
print("\n💾 Saving version 1...")
vm.save_version(
    df_sample, 
    name='bike_availability',
    description='Initial dataset with 30 days of data'
)

# Make some changes and save version 2
df_modified = df_sample.copy()
df_modified['total_capacity'] = df_modified['bikes_available'] + df_modified['docks_available']

print("\n💾 Saving version 2...")
vm.save_version(
    df_modified,
    name='bike_availability',
    description='Added total_capacity column'
)

# List all versions
vm.list_versions()

---

### 🧠 Task 4.1: Extend Version Manager (50% Scaffolding)

**Your Task**: Add a `delete_version()` method to the DataVersionManager class.

**What you'll learn**:
- Extending existing classes
- File system operations with pathlib
- Data lifecycle management

**Requirements**:
1. Implement method that deletes a specific version
2. Remove file from disk
3. Update metadata
4. Add error handling
5. Test by deleting version 1

**Method signature provided** - Implement the logic:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 5.1 SOLUTION: EXTEND VERSION MANAGER
# ═══════════════════════════════════════════════════════════

# Extended DataVersionManager with delete capability
class DataVersionManagerExtended(DataVersionManager):
    """
    Extended version with delete_version capability.
    """
    
    def delete_version(self, name: str, version: int):
        """
        Delete a specific version of a dataset.
        
        Parameters:
        -----------
        name : str
            Dataset name
        version : int
            Version number to delete
        """
        # Find the version in metadata
        version_info = next(
            (v for v in self.metadata['versions'] 
             if v['name'] == name and v['version'] == version),
            None
        )
        
        # Check if version exists
        if version_info is None:
            raise ValueError(f"Version {version} of '{name}' not found")
        
        # Get filepath
        filepath = self.base_dir / version_info['filename']
        
        # Delete the file from disk
        try:
            if filepath.exists():
                filepath.unlink()
                print(f"🗑️  Deleted file: {version_info['filename']}")
            else:
                print(f"⚠️  File not found: {version_info['filename']}")
        except Exception as e:
            raise IOError(f"Failed to delete file: {e}")
        
        # Remove from metadata
        self.metadata['versions'] = [
            v for v in self.metadata['versions']
            if not (v['name'] == name and v['version'] == version)
        ]
        self._save_metadata()
        
        # Print confirmation
        print(f"✅ Successfully deleted version {version} of '{name}'")
        print(f"   File: {version_info['filename']}")
        print(f"   Size freed: {version_info['size_bytes']/1024:.1f} KB")


# Test the extended version manager
print("🧪 Testing delete_version functionality...\n")

# Create new extended manager
vm_ext = DataVersionManagerExtended(TEST_DIR / 'test_versions')

# Save a few versions for testing
print("💾 Creating test versions:")
df_test = df_sample.head(100).copy()

vm_ext.save_version(df_test, 'test_dataset', 'Version 1 - Original')
df_test['test_col'] = 'modified'
vm_ext.save_version(df_test, 'test_dataset', 'Version 2 - Added test_col')
df_test['another_col'] = 42
vm_ext.save_version(df_test, 'test_dataset', 'Version 3 - Added another_col')

print("\n📚 Versions before deletion:")
vm_ext.list_versions('test_dataset')

# Delete version 1
print("\n🗑️  Deleting version 1...")
try:
    vm_ext.delete_version('test_dataset', 1)
except Exception as e:
    print(f"❌ Error: {e}")

print("\n📚 Versions after deletion:")
vm_ext.list_versions('test_dataset')

# Verify file is gone
print(f"\n🔍 Verification:")
print(f"   Version 1 file exists: {any(vm_ext.base_dir.glob('test_dataset_v1_*.parquet'))}")
print(f"   Version 2 file exists: {any(vm_ext.base_dir.glob('test_dataset_v2_*.parquet'))}")
print(f"   Version 3 file exists: {any(vm_ext.base_dir.glob('test_dataset_v3_*.parquet'))}")

print("\n🎓 Learning Points:")
print("  ✓ Always validate before deleting")
print("  ✓ Update both filesystem and metadata")
print("  ✓ Handle errors gracefully")
print("  ✓ Consider adding confirmation prompts for production use")

---

## 📝 Part 6: Data Documentation

### Documentation Best Practices

Always document:
1. **Source** - Where did the data come from?
2. **Date** - When was it collected?
3. **Transformations** - What changes were made?
4. **Schema** - What do the columns mean?
5. **Quality** - Are there known issues?

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. DATA DOCUMENTATION
# ═══════════════════════════════════════════════════════════

def create_data_documentation(df, dataset_name, source, description):
    """
    Create comprehensive data documentation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Dataset to document
    dataset_name : str
        Name of the dataset
    source : str
        Data source
    description : str
        Dataset description
    
    Returns:
    --------
    dict : Documentation metadata
    """
    doc = {
        'dataset_name': dataset_name,
        'description': description,
        'source': source,
        'created_date': datetime.now().isoformat(),
        'shape': {
            'rows': len(df),
            'columns': len(df.columns)
        },
        'columns': {},
        'data_types': {},
        'missing_values': {},
        'date_range': {},
        'file_info': {}
    }
    
    # Document each column
    for col in df.columns:
        doc['columns'][col] = {
            'dtype': str(df[col].dtype),
            'non_null_count': int(df[col].notna().sum()),
            'null_count': int(df[col].isna().sum()),
            'null_percentage': float(df[col].isna().sum() / len(df) * 100)
        }
        
        # Add statistics for numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            doc['columns'][col].update({
                'min': float(df[col].min()),
                'max': float(df[col].max()),
                'mean': float(df[col].mean()),
                'median': float(df[col].median()),
                'std': float(df[col].std())
            })
        
        # Add unique values for categorical columns
        if df[col].dtype == 'object' or df[col].nunique() < 50:
            doc['columns'][col]['unique_values'] = int(df[col].nunique())
            doc['columns'][col]['sample_values'] = df[col].dropna().unique()[:5].tolist()
    
    # Document date range if timestamp column exists
    if 'timestamp' in df.columns:
        doc['date_range'] = {
            'start': df['timestamp'].min().isoformat(),
            'end': df['timestamp'].max().isoformat(),
            'duration_days': (df['timestamp'].max() - df['timestamp'].min()).days
        }
    
    return doc


# Create documentation for our sample data
print("📝 Creating data documentation...")
documentation = create_data_documentation(
    df_sample,
    dataset_name='amsterdam_bike_availability',
    source='Simulated data for demonstration',
    description='Bike availability data with weather information for Amsterdam stations'
)

# Save documentation
doc_file = TEST_DIR / 'bike_availability_documentation.json'
with open(doc_file, 'w') as f:
    json.dump(documentation, f, indent=2, default=str)

print(f"✅ Documentation saved: {doc_file}")

# Display summary
print("\n" + "=" * 60)
print("📊 Dataset Documentation Summary:")
print("=" * 60)
print(f"Dataset: {documentation['dataset_name']}")
print(f"Description: {documentation['description']}")
print(f"Rows: {documentation['shape']['rows']:,}")
print(f"Columns: {documentation['shape']['columns']}")
print(f"\nDate Range: {documentation['date_range']['start'][:10]} to {documentation['date_range']['end'][:10]}")
print(f"Duration: {documentation['date_range']['duration_days']} days")

print("\n📋 Column Summary:")
for col, info in list(documentation['columns'].items())[:5]:  # Show first 5 columns
    print(f"\n  {col}:")
    print(f"    Type: {info['dtype']}")
    print(f"    Non-null: {info['non_null_count']:,}")
    if 'mean' in info:
        print(f"    Mean: {info['mean']:.2f}")

---

### 🧠 Task 6.1: Create Custom Documentation (40% Scaffolding)

**Your Task**: Document your latest dataset version with README and metadata.

**What you'll learn**:
- Professional data documentation standards
- Schema documentation best practices
- Making data discoverable

**Requirements**:
1. Load latest version from version manager
2. Create detailed README.md file
3. Generate metadata JSON with statistics
4. Document all transformations applied

**Template provided** - Customize for your data:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 6.1 SOLUTION: CUSTOM DOCUMENTATION
# ═══════════════════════════════════════════════════════════

# Load the latest version
df_latest = vm.load_version('bike_availability')

# Create comprehensive README
readme_content = f"""# Amsterdam Bike Availability Dataset

## Overview
This dataset contains bike availability data for Amsterdam stations with integrated weather information.
The data includes real-time bike counts, dock availability, and corresponding weather conditions.

## Source
- **Bike Data**: CityBikes API (simulated for demonstration)
- **Weather Data**: Open-Meteo API (simulated)
- **Collection Period**: {df_latest['timestamp'].min().strftime('%Y-%m-%d')} to {df_latest['timestamp'].max().strftime('%Y-%m-%d')}
- **Update Frequency**: Hourly snapshots
- **Coverage**: {df_latest['station_id'].nunique()} stations across Amsterdam

## Schema

| Column | Type | Description | Example |
|--------|------|-------------|---------|
| timestamp | datetime64 | Time of observation | 2026-01-01 00:00:00 |
| station_id | object | Unique station identifier | Station_001 |
| station_name | object | Human-readable station name | Amsterdam Station_001 |
| bikes_available | int64 | Number of bikes available | 15 |
| docks_available | int64 | Number of empty docks | 5 |
| latitude | float64 | Station latitude | 52.3702 |
| longitude | float64 | Station longitude | 4.8952 |
| temperature_c | float64 | Temperature in Celsius | 12.5 |
| precipitation_mm | float64 | Precipitation in millimeters | 0.0 |
| wind_speed_kmh | float64 | Wind speed in km/h | 15.3 |
| total_capacity | int64 | Total station capacity | 20 |

## Data Quality

- **Completeness**: {100 - df_latest.isnull().sum().sum() / (len(df_latest) * len(df_latest.columns)) * 100:.1f}% complete
- **Missing values**: {df_latest.isnull().sum().sum()} total null values
- **Outliers**: Values clipped to realistic ranges
- **Validation**: All bike/dock counts ≥ 0

## Transformations Applied

1. **Data Generation**: Created synthetic hourly data with realistic patterns
2. **Time patterns**: Simulated commute hour peaks (8-9 AM, 5-6 PM)
3. **Weekend adjustment**: Reduced availability by 30% on weekends
4. **Weather integration**: Added correlated weather data
5. **Capacity calculation**: Added total_capacity = bikes_available + docks_available
6. **Validation**: Clipped negative values to zero

## Statistics

- **Total Records**: {len(df_latest):,}
- **Time Span**: {(df_latest['timestamp'].max() - df_latest['timestamp'].min()).days} days
- **Stations**: {df_latest['station_id'].nunique()}
- **Average Bikes per Station**: {df_latest['bikes_available'].mean():.1f}
- **Average Temperature**: {df_latest['temperature_c'].mean():.1f}°C

## Usage Example

```python
import pandas as pd

# Load data
df = pd.read_parquet('bike_availability_v2_latest.parquet')

# Basic analysis
print(f"Total observations: {{len(df):,}}")
print(f"Average bikes: {{df['bikes_available'].mean():.1f}}")

# Filter by station
station_data = df[df['station_id'] == 'Station_001']

# Time series analysis
df.set_index('timestamp').resample('D')['bikes_available'].mean().plot()
```

## Data Governance

- **Version**: 2
- **Created**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- **Format**: Parquet (compressed)
- **Size**: {df_latest.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB (in memory)
- **Maintainer**: Course Team
- **License**: Educational Use Only

## Known Issues

- Data is synthetically generated for educational purposes
- Weather correlations are simplified
- Station locations are randomized within Amsterdam bounds

## Contact

For questions or issues with this dataset:
- **Repository**: bike-availability-data-science
- **Module**: M2_03 Data Storage Patterns
- **Last Updated**: {datetime.now().strftime('%Y-%m-%d')}

---
*This dataset was created as part of the Bike Availability Data Science course.*
"""

# Save README
readme_path = TEST_DIR / 'bike_availability_README.md'
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"✅ Created README: {readme_path.name}")

# Create metadata JSON with comprehensive statistics
metadata = {
    'dataset_name': 'amsterdam_bike_availability',
    'version': 2,
    'created_date': datetime.now().isoformat(),
    'format': 'parquet',
    'compression': 'snappy',
    
    # Basic stats
    'row_count': len(df_latest),
    'column_count': len(df_latest.columns),
    'memory_usage_mb': df_latest.memory_usage(deep=True).sum() / 1024 / 1024,
    
    # Schema
    'columns': list(df_latest.columns),
    'data_types': {col: str(dtype) for col, dtype in df_latest.dtypes.items()},
    
    # Missing values
    'missing_values': {
        col: int(df_latest[col].isnull().sum()) 
        for col in df_latest.columns
    },
    'missing_percentage': {
        col: float(df_latest[col].isnull().sum() / len(df_latest) * 100)
        for col in df_latest.columns
    },
    
    # Numeric statistics
    'numeric_statistics': {},
    
    # Categorical statistics
    'categorical_statistics': {}
}

# Add numeric column statistics
for col in df_latest.select_dtypes(include=['number']).columns:
    metadata['numeric_statistics'][col] = {
        'min': float(df_latest[col].min()),
        'max': float(df_latest[col].max()),
        'mean': float(df_latest[col].mean()),
        'median': float(df_latest[col].median()),
        'std': float(df_latest[col].std()),
        'q25': float(df_latest[col].quantile(0.25)),
        'q75': float(df_latest[col].quantile(0.75))
    }

# Add categorical column statistics
for col in df_latest.select_dtypes(include=['object']).columns:
    metadata['categorical_statistics'][col] = {
        'unique_values': int(df_latest[col].nunique()),
        'most_common': df_latest[col].mode()[0] if len(df_latest[col].mode()) > 0 else None,
        'sample_values': df_latest[col].dropna().unique()[:5].tolist()
    }

# Add time range info
if 'timestamp' in df_latest.columns:
    metadata['time_range'] = {
        'start': df_latest['timestamp'].min().isoformat(),
        'end': df_latest['timestamp'].max().isoformat(),
        'duration_days': (df_latest['timestamp'].max() - df_latest['timestamp'].min()).days,
        'frequency': 'hourly'
    }

# Save metadata
metadata_path = TEST_DIR / 'bike_availability_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"✅ Created metadata: {metadata_path.name}")

# Print summary
print("\n" + "="*70)
print("📊 SOLUTION: Documentation Summary")
print("="*70)
print(f"\n✅ Documentation created:")
print(f"   📄 README.md ({readme_path.stat().st_size:,} bytes)")
print(f"   📄 metadata.json ({metadata_path.stat().st_size:,} bytes)")

print(f"\n📋 Dataset Overview:")
print(f"   • Name: {metadata['dataset_name']}")
print(f"   • Version: {metadata['version']}")
print(f"   • Rows: {metadata['row_count']:,}")
print(f"   • Columns: {metadata['column_count']}")
print(f"   • Completeness: {100 - sum(metadata['missing_values'].values()) / (metadata['row_count'] * metadata['column_count']) * 100:.1f}%")

print(f"\n🎓 Learning Points:")
print("  ✓ Comprehensive documentation makes data discoverable")
print("  ✓ Include both human-readable (MD) and machine-readable (JSON) formats")
print("  ✓ Document transformations, data quality, and known issues")
print("  ✓ Provide usage examples for different audiences")

---

## 🔒 Part 7: .gitignore Best Practices

### What NOT to Commit to Git

- ❌ Large data files (> 100MB)
- ❌ Raw data downloads
- ❌ API keys and credentials
- ❌ Personal/sensitive information
- ❌ Temporary/cache files

### What TO Commit

- ✅ Code and notebooks
- ✅ Small sample datasets (< 10MB)
- ✅ Data acquisition scripts
- ✅ Documentation and metadata
- ✅ README files explaining data sources

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. GITIGNORE RECOMMENDATIONS
# ═══════════════════════════════════════════════════════════

gitignore_content = """# Data Science Project .gitignore

# Large Data Files
data/raw/*.csv
data/raw/*.json
data/raw/*.parquet
data/processed/*.csv
data/processed/*.parquet

# Keep sample data and documentation
!data/raw/sample_*.csv
!data/raw/README.md
!data/processed/README.md

# API Keys and Credentials
.env
.env.local
*.key
credentials.json
secrets.json

# Jupyter Notebook Checkpoints
.ipynb_checkpoints/
*/.ipynb_checkpoints/*

# Python Cache
__pycache__/
*.py[cod]
*$py.class
*.so

# Virtual Environments
venv/
env/
ENV/
.venv

# IDE Settings
.vscode/
.idea/
*.swp
*.swo
*~

# OS Files
.DS_Store
Thumbs.db
desktop.ini

# Model Artifacts (large files)
models/*.h5
models/*.pkl
models/*.joblib

# Temporary Files
*.tmp
*.temp
temp_*

# Large Results
results/*.png
results/*.jpg
!results/summary_*.png  # Keep summary images
"""

print("📝 Recommended .gitignore content:")
print("=" * 60)
print(gitignore_content)
print("=" * 60)

# Check if .gitignore exists in project root
gitignore_path = Path(project_root) / '.gitignore'
if gitignore_path.exists():
    print(f"\n✅ .gitignore file exists at: {gitignore_path}")
    print("   Review and update it with the recommendations above")
else:
    print(f"\n⚠️ No .gitignore file found at: {gitignore_path}")
    print("   Consider creating one with the recommendations above")

---

## 📊 Part 8: Data Catalog

Create a data catalog to track all datasets in your project.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 11. DATA CATALOG
# ═══════════════════════════════════════════════════════════

class DataCatalog:
    """
    Manage a catalog of all datasets in the project.
    """
    
    def __init__(self, catalog_file):
        self.catalog_file = Path(catalog_file)
        self.catalog = self._load_catalog()
    
    def _load_catalog(self):
        """Load catalog from file."""
        if self.catalog_file.exists():
            with open(self.catalog_file, 'r') as f:
                return json.load(f)
        return {'datasets': []}
    
    def _save_catalog(self):
        """Save catalog to file."""
        with open(self.catalog_file, 'w') as f:
            json.dump(self.catalog, f, indent=2)
    
    def register_dataset(self, name, path, description, source, 
                        tags=None, metadata=None):
        """Register a new dataset in the catalog."""
        
        path = Path(path)
        
        dataset_info = {
            'name': name,
            'path': str(path),
            'description': description,
            'source': source,
            'registered_date': datetime.now().isoformat(),
            'tags': tags or [],
            'metadata': metadata or {}
        }
        
        # Add file information if file exists
        if path.exists():
            dataset_info['file_info'] = {
                'size_bytes': path.stat().st_size,
                'size_mb': path.stat().st_size / 1024 / 1024,
                'modified_date': datetime.fromtimestamp(path.stat().st_mtime).isoformat()
            }
        
        # Update or add dataset
        existing_idx = next((i for i, d in enumerate(self.catalog['datasets']) 
                           if d['name'] == name), None)
        
        if existing_idx is not None:
            self.catalog['datasets'][existing_idx] = dataset_info
            print(f"✅ Updated dataset '{name}' in catalog")
        else:
            self.catalog['datasets'].append(dataset_info)
            print(f"✅ Registered new dataset '{name}' in catalog")
        
        self._save_catalog()
    
    def list_datasets(self, tag=None):
        """List all datasets or filter by tag."""
        datasets = self.catalog['datasets']
        
        if tag:
            datasets = [d for d in datasets if tag in d.get('tags', [])]
        
        if not datasets:
            print(f"No datasets found{' with tag: ' + tag if tag else ''}")
            return
        
        print(f"\n📚 Data Catalog{' (tag: ' + tag + ')' if tag else ''}:")
        print("=" * 100)
        print(f"{'Name':<25} {'Source':<20} {'Size (MB)':<12} {'Tags':<20}")
        print("-" * 100)
        
        for d in datasets:
            size_mb = d.get('file_info', {}).get('size_mb', 0)
            tags_str = ', '.join(d.get('tags', []))
            print(f"{d['name']:<25} {d['source']:<20} {size_mb:>10.2f}  {tags_str:<20}")
        
        print("-" * 100)
        print(f"Total datasets: {len(datasets)}")
    
    def get_dataset(self, name):
        """Get information about a specific dataset."""
        dataset = next((d for d in self.catalog['datasets'] if d['name'] == name), None)
        
        if dataset is None:
            print(f"❌ Dataset '{name}' not found in catalog")
            return None
        
        print(f"\n📊 Dataset: {dataset['name']}")
        print("=" * 60)
        print(f"Description: {dataset['description']}")
        print(f"Source: {dataset['source']}")
        print(f"Path: {dataset['path']}")
        print(f"Registered: {dataset['registered_date'][:19]}")
        print(f"Tags: {', '.join(dataset.get('tags', []))}")
        
        if 'file_info' in dataset:
            print(f"\nFile Info:")
            print(f"  Size: {dataset['file_info']['size_mb']:.2f} MB")
            print(f"  Modified: {dataset['file_info']['modified_date'][:19]}")
        
        return dataset


# Example usage
catalog_file = TEST_DIR / 'data_catalog.json'
catalog = DataCatalog(catalog_file)

# Register some datasets
catalog.register_dataset(
    name='bike_availability_v1',
    path=TEST_DIR / 'bike_availability_v1_2026-01-15_143022.parquet',
    description='Initial bike availability dataset with 30 days of data',
    source='CityBikes API',
    tags=['raw', 'bike', 'amsterdam']
)

catalog.register_dataset(
    name='bike_availability_v2',
    path=TEST_DIR / 'bike_availability_v2_2026-01-15_143023.parquet',
    description='Bike availability with total_capacity column added',
    source='CityBikes API (processed)',
    tags=['processed', 'bike', 'amsterdam', 'featured']
)

# List all datasets
catalog.list_datasets()

# List datasets by tag
print("\n")
catalog.list_datasets(tag='processed')

# Get specific dataset info
catalog.get_dataset('bike_availability_v2')

---

## 💾 Part 9: Save Best Practices Example

Here's a complete example of saving data with all best practices.

In [ ]:
# ═══════════════════════════════════════════════════════════
# 12. COMPLETE SAVE EXAMPLE
# ═══════════════════════════════════════════════════════════

def save_dataset_with_best_practices(df, dataset_name, source, description, 
                                    output_dir, version_manager=None, 
                                    data_catalog=None):
    """
    Save dataset following all best practices.
    
    This function:
    1. Saves data in Parquet format
    2. Creates documentation
    3. Manages versioning
    4. Updates data catalog
    5. Returns file paths
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save data in Parquet format
    timestamp = datetime.now().strftime('%Y-%m-%d_%H%M%S')
    data_file = output_dir / f"{dataset_name}_{timestamp}.parquet"
    df.to_parquet(data_file, index=False)
    print(f"✅ Saved data: {data_file.name}")
    
    # 2. Create and save documentation
    doc = create_data_documentation(df, dataset_name, source, description)
    doc_file = data_file.with_suffix('.metadata.json')
    with open(doc_file, 'w') as f:
        json.dump(doc, f, indent=2, default=str)
    print(f"✅ Saved documentation: {doc_file.name}")
    
    # 3. Version management (if provided)
    if version_manager:
        version_manager.save_version(df, dataset_name, description)
        print(f"✅ Version tracked")
    
    # 4. Update data catalog (if provided)
    if data_catalog:
        data_catalog.register_dataset(
            name=dataset_name,
            path=data_file,
            description=description,
            source=source,
            tags=['processed', 'documented']
        )
        print(f"✅ Catalog updated")
    
    # 5. Create README if it doesn't exist
    readme_file = output_dir / 'README.md'
    if not readme_file.exists():
        readme_content = f"""# Processed Data

This directory contains processed datasets ready for analysis.

## Latest Datasets

### {dataset_name}
- **Source**: {source}
- **Description**: {description}
- **Created**: {timestamp}
- **Rows**: {len(df):,}
- **Columns**: {len(df.columns)}

## File Naming Convention

Files follow the pattern: `{{dataset_name}}_{{YYYY-MM-DD_HHMMSS}}.parquet`

## Documentation

Each dataset has an accompanying `.metadata.json` file with:
- Column descriptions and statistics
- Data quality information
- Source and transformation details
- Date range and sample size

Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
        with open(readme_file, 'w') as f:
            f.write(readme_content)
        print(f"✅ Created README: {readme_file.name}")
    
    return {
        'data_file': data_file,
        'doc_file': doc_file,
        'readme_file': readme_file
    }


# Example: Save with all best practices
print("💾 Saving dataset with best practices...")
print("=" * 60)

files = save_dataset_with_best_practices(
    df=df_sample,
    dataset_name='bike_availability_clean',
    source='CityBikes API + Open-Meteo API',
    description='Cleaned and merged bike availability with weather data',
    output_dir=TEST_DIR,
    version_manager=vm,
    data_catalog=catalog
)

print("\n" + "=" * 60)
print("📁 Files created:")
for key, path in files.items():
    print(f"  • {key}: {path.name}")

---

### 🧠 Task 9.1: Build Complete Workflow (30% Scaffolding)

**Your Task**: Create an end-to-end data storage workflow for a new scenario.

**What you'll learn**:
- Integrating all storage concepts
- Making architectural decisions
- Building production-ready solutions

**Scenario**: You've collected 7 days of bike data for 3 stations. Build a complete storage solution.

**Requirements**:
1. Generate sample data (3 stations × 7 days × 24 hours)
2. Choose and justify file format
3. Implement versioning strategy
4. Create complete documentation
5. Set up logical directory structure
6. Write decision summary

**Minimal scaffolding** - Design your own approach:

In [ ]:
# ═══════════════════════════════════════════════════════════
# TASK 9.1 SOLUTION: COMPLETE WORKFLOW
# ═══════════════════════════════════════════════════════════

"""
Complete end-to-end storage workflow implementation.

Scenario: Store 7 days of bike data for 3 stations with full best practices.
"""

print("🚀 Starting Complete Data Storage Workflow")
print("="*70)

# Step 1: Generate sample data
print("\n📊 Step 1: Generating Sample Data...")
print("-"*70)

# Create 3 stations × 7 days × 24 hours = 504 records
np.random.seed(42)
stations_list = ['Station_A', 'Station_B', 'Station_C']
hours_weekly = pd.date_range('2026-01-15', periods=24*7, freq='H')

data_weekly = []
for station in stations_list:
    for timestamp in hours_weekly:
        hour = timestamp.hour
        day_of_week = timestamp.dayofweek
        
        # Realistic patterns
        base_bikes = 12
        if hour in [8, 9, 17, 18]:  # Rush hours
            base_bikes += 8
        if day_of_week >= 5:  # Weekend
            base_bikes = int(base_bikes * 0.6)
        
        data_weekly.append({
            'timestamp': timestamp,
            'station_id': station,
            'bikes_available': max(0, base_bikes + np.random.randint(-4, 5)),
            'docks_available': max(0, 20 - base_bikes + np.random.randint(-3, 4)),
            'temperature_c': 10 + np.random.uniform(-5, 8),
            'precipitation_mm': max(0, np.random.exponential(0.3)),
            'wind_speed_kmh': max(0, np.random.normal(12, 4))
        })

df_weekly = pd.DataFrame(data_weekly)

print(f"✅ Generated {len(df_weekly):,} records")
print(f"   • Stations: {len(stations_list)}")
print(f"   • Days: 7")
print(f"   • Frequency: Hourly")
print(f"   • Date range: {df_weekly['timestamp'].min()} to {df_weekly['timestamp'].max()}")

# Step 2: Choose format with justification
print("\n🗂️  Step 2: Format Selection Analysis...")
print("-"*70)

# Test save performance for each format
format_analysis = {}
for fmt in ['CSV', 'JSON', 'Parquet']:
    test_path = TEST_DIR / f'workflow_test.{fmt.lower()}'
    
    start = time.time()
    if fmt == 'CSV':
        df_weekly.to_csv(test_path, index=False)
    elif fmt == 'JSON':
        df_weekly.to_json(test_path, orient='records', date_format='iso')
    elif fmt == 'Parquet':
        df_weekly.to_parquet(test_path, index=False, compression='snappy')
    
    save_time = time.time() - start
    file_size_mb = test_path.stat().st_size / 1024 / 1024
    
    format_analysis[fmt] = {
        'save_time': save_time,
        'size_mb': file_size_mb
    }
    test_path.unlink()  # Cleanup
    
    print(f"   {fmt:8s}: {file_size_mb:.2f} MB, {save_time:.3f}s")

# Make decision
chosen_format = 'Parquet'
print(f"\n✅ Selected Format: {chosen_format}")
print(f"   Justification:")
print(f"   • Best compression ({format_analysis['Parquet']['size_mb']:.2f} MB vs {format_analysis['CSV']['size_mb']:.2f} MB CSV)")
print(f"   • Fast I/O ({format_analysis['Parquet']['save_time']:.3f}s)")
print(f"   • Preserves data types (important for timestamps)")
print(f"   • Industry standard for analytics")

# Step 3: Implement versioning
print("\n🔢 Step 3: Version Management...")
print("-"*70)

# Create dedicated workflow directory
workflow_dir = TEST_DIR / 'weekly_bike_workflow'
workflow_vm = DataVersionManager(workflow_dir)

# Save initial version
version_info = workflow_vm.save_version(
    df_weekly,
    name='weekly_bikes',
    description='7 days of bike data for 3 stations - initial collection'
)

print(f"✅ Versioning implemented")
print(f"   Strategy: Timestamp-based with metadata tracking")
print(f"   Version file: {version_info['filename']}")

# Step 4: Create comprehensive documentation
print("\n📝 Step 4: Documentation Creation...")
print("-"*70)

# README
workflow_readme = f"""# Weekly Bike Availability Dataset

**Purpose**: Track bike availability for 3 key Amsterdam stations over weekly periods.

## Quick Stats
- **Records**: {len(df_weekly):,}
- **Stations**: {', '.join(stations_list)}
- **Period**: 7 days (hourly snapshots)
- **Format**: Parquet with Snappy compression
- **Size**: {version_info['size_bytes']/1024:.1f} KB

## Columns
- `timestamp`: Observation time (hourly)
- `station_id`: Station identifier (A, B, or C)
- `bikes_available`: Available bikes count
- `docks_available`: Available docks count
- `temperature_c`, `precipitation_mm`, `wind_speed_kmh`: Weather data

## Usage
```python
import pandas as pd
df = pd.read_parquet('{version_info['filename']}')
```

## Update Schedule
- Weekly refresh
- Automated versioning
- Keep last 4 versions (1 month)

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

workflow_readme_path = workflow_dir / 'README.md'
with open(workflow_readme_path, 'w') as f:
    f.write(workflow_readme)

# Metadata
workflow_metadata = {
    'dataset': 'weekly_bikes',
    'version': 1,
    'created': datetime.now().isoformat(),
    'rows': len(df_weekly),
    'columns': list(df_weekly.columns),
    'stations': stations_list,
    'date_range': {
        'start': df_weekly['timestamp'].min().isoformat(),
        'end': df_weekly['timestamp'].max().isoformat()
    }
}

workflow_metadata_path = workflow_dir / 'metadata.json'
with open(workflow_metadata_path, 'w') as f:
    json.dump(workflow_metadata, f, indent=2)

print(f"✅ Documentation created")
print(f"   • README.md")
print(f"   • metadata.json")

# Step 5: Directory structure
print("\n📁 Step 5: Directory Structure...")
print("-"*70)

# Create organized structure
(workflow_dir / 'archive').mkdir(exist_ok=True)
(workflow_dir / 'docs').mkdir(exist_ok=True)

# Move documentation
import shutil
shutil.copy(workflow_readme_path, workflow_dir / 'docs' / 'README.md')

print(f"✅ Directory structure established:")
print(f"   {workflow_dir.name}/")
print(f"   ├── {version_info['filename']}")
print(f"   ├── versions.json")
print(f"   ├── README.md")
print(f"   ├── metadata.json")
print(f"   ├── archive/  (for old versions)")
print(f"   └── docs/     (documentation)")

# Step 6: Summary report
print("\n" + "="*70)
print("📊 WORKFLOW SUMMARY REPORT")
print("="*70)

print(f"""
✅ IMPLEMENTATION COMPLETE

1. DATA GENERATION
   • Created: {len(df_weekly):,} records
   • Stations: {len(stations_list)} ({', '.join(stations_list)})
   • Timespan: 7 days (hourly)
   • Patterns: Rush hour peaks, weekend adjustments

2. FORMAT DECISION
   • Chosen: Parquet with Snappy compression
   • Size: {format_analysis['Parquet']['size_mb']:.2f} MB
   • Compression: {(1 - format_analysis['Parquet']['size_mb']/format_analysis['CSV']['size_mb'])*100:.0f}% vs CSV
   • Rationale: Best balance of size, speed, and type preservation

3. VERSIONING STRATEGY
   • System: Custom DataVersionManager
   • Naming: {{dataset}}_v{{N}}_{{timestamp}}.parquet
   • Metadata: Tracked in versions.json
   • Retention: Keep last 4 versions

4. DOCUMENTATION
   • README.md: Human-readable overview
   • metadata.json: Machine-readable specifications
   • Inline: Column descriptions and usage examples
   • Quality: Data validation notes included

5. DIRECTORY STRUCTURE
   • Organized: Data, docs, and archive separated
   • Scalable: Easy to add more stations/periods
   • Maintainable: Clear naming conventions

6. PRODUCTION READINESS
   • ✓ Automated versioning
   • ✓ Comprehensive documentation
   • ✓ Efficient storage format
   • ✓ Clear organization
   • ✓ Easy to integrate with pipelines

NEXT STEPS FOR PRODUCTION:
   → Add automated tests for data quality
   → Implement backup strategy (cloud storage)
   → Set up monitoring for file sizes
   → Create update schedule/pipeline
   → Add access control and audit logging

🎓 KEY LEARNINGS:
   • Planning prevents problems
   • Documentation is not optional
   • Format choice matters at scale
   • Version everything you care about
   • Think about the full lifecycle upfront
""")

print("="*70)
print("✅ Complete workflow demonstration finished!")
print("="*70)

---

## 🔥 Part 9a: Optional Advanced Challenges

**For advanced learners**: These challenges explore production-grade storage patterns!

### Challenge 6.1: DVC Integration 🔄

**Task**: Set up Data Version Control (DVC) for the project.

**Requirements**:
- Install DVC: `pip install dvc`
- Initialize DVC in the project
- Track the processed data directory with DVC
- Create a DVC pipeline for data processing
- Push data to remote storage (local for testing)

**Hints**:
```bash
dvc init
dvc add data/processed
git add data/processed.dvc .gitignore
dvc remote add -d local_remote /tmp/dvc-storage
dvc push
```

**Learning**: Enterprise-level data versioning

---

### Challenge 6.2: Cloud Storage Integration ☁️

**Task**: Implement cloud storage (AWS S3 or Google Cloud Storage) for large datasets.

**Requirements**:
- Set up credentials for cloud provider
- Create function to upload/download from cloud
- Implement caching: check local first, download if needed
- Add cloud paths to data catalog
- Handle connection errors gracefully

**Hints**:
- Use `boto3` for AWS or `google-cloud-storage`
- Consider `s3fs` or `gcsfs` for Parquet direct reading
- Cache metadata locally, data in cloud

**Learning**: Scalable storage for production systems

---

### Challenge 6.3: Streaming Data Storage 📡

**Task**: Design a storage system for real-time streaming bike data.

**Requirements**:
- Create append-only storage structure
- Implement time-partitioned files (hourly/daily)
- Build efficient query interface for date ranges
- Handle late-arriving data
- Implement data compaction strategy

**Suggested structure**:
```
data/
  streaming/
    2026/
      01/
        bike_data_2026-01-01_00.parquet
        bike_data_2026-01-01_01.parquet
        ...
```

**Hints**:
- Use Parquet partitioning: `df.to_parquet('data/streaming', partition_cols=['year', 'month', 'day'])`
- Consider: How to query last 24 hours efficiently?
- Think about: When to compact small files?

**Learning**: Real-time data engineering patterns

---

### Challenge 6.4: Data Lineage Tracker 📊

**Task**: Build a system to track data transformations and dependencies.

**Requirements**:
- Create `DataLineage` class that records:
  - Input datasets used
  - Transformations applied
  - Output datasets created
  - Timestamps and user
- Generate visual lineage graph
- Export lineage as JSON
- Integrate with your save workflow

**Example tracking**:
```python
lineage = DataLineage()
lineage.add_input('bike_api_data.csv')
lineage.add_transformation('remove_nulls', params={'threshold': 0.1})
lineage.add_transformation('add_bikeability_score')
lineage.add_output('bike_processed_v2.parquet')
lineage.visualize()  # Creates flowchart
```

**Hints**:
- Use `graphviz` or `networkx` for visualization
- Store lineage metadata with datasets
- Consider: How to trace data quality issues back to source?

**Learning**: Data governance and reproducibility

---

## 🔥 Part 9b: Challenge Solutions

Complete implementations for all advanced challenges!

### Challenge 6.1 Solution: DVC Integration 🔄

In [ ]:
# ═══════════════════════════════════════════════════════════
# CHALLENGE 6.1 SOLUTION: DVC INTEGRATION
# ═══════════════════════════════════════════════════════════

"""
Data Version Control (DVC) is Git for data - tracks large files outside Git
while maintaining version control capabilities.

This solution demonstrates:
1. DVC initialization and configuration
2. Tracking datasets with DVC
3. Creating a simple DVC pipeline
4. Remote storage setup (local for testing)
"""

import subprocess
import shutil
from pathlib import Path

print("🔄 CHALLENGE 6.1 SOLUTION: DVC Integration")
print("="*70)

# Create test directory for DVC demo
dvc_demo_dir = TEST_DIR / 'dvc_demo'
dvc_demo_dir.mkdir(exist_ok=True)

print(f"\n📁 Working directory: {dvc_demo_dir}")

# Step 1: Check if DVC is installed
print("\n1️⃣ Checking DVC installation...")
try:
    result = subprocess.run(['dvc', 'version'], 
                          capture_output=True, text=True, cwd=dvc_demo_dir)
    if result.returncode == 0:
        print(f"   ✅ DVC installed: {result.stdout.split()[0]}")
    else:
        print("   ⚠️  DVC not found. Install with: pip install dvc")
        print("   For this demo, we'll show the commands without executing.")
except FileNotFoundError:
    print("   ⚠️  DVC not installed. This is optional - showing demo commands.")
    print("   Install with: pip install 'dvc[s3]' or pip install 'dvc[gs]'")

# Step 2: Initialize Git (required for DVC)
print("\n2️⃣ Initializing Git repository...")
git_dir = dvc_demo_dir / '.git'
if not git_dir.exists():
    try:
        subprocess.run(['git', 'init'], cwd=dvc_demo_dir, check=True, 
                      capture_output=True)
        subprocess.run(['git', 'config', 'user.email', 'demo@example.com'],
                      cwd=dvc_demo_dir, check=True, capture_output=True)
        subprocess.run(['git', 'config', 'user.name', 'Demo User'],
                      cwd=dvc_demo_dir, check=True, capture_output=True)
        print("   ✅ Git initialized")
    except:
        print("   ⚠️  Git init failed (demo purposes)")
else:
    print("   ✅ Git already initialized")

# Step 3: Initialize DVC
print("\n3️⃣ Initializing DVC...")
dvc_dir = dvc_demo_dir / '.dvc'
if not dvc_dir.exists():
    try:
        subprocess.run(['dvc', 'init'], cwd=dvc_demo_dir, check=True,
                      capture_output=True)
        print("   ✅ DVC initialized")
        print("   Created: .dvc/ directory with DVC config")
    except:
        print("   ⚠️  DVC init failed - showing manual setup")
        dvc_dir.mkdir(exist_ok=True)

# Step 4: Create sample data to track
print("\n4️⃣ Creating sample dataset...")
data_dir = dvc_demo_dir / 'data'
data_dir.mkdir(exist_ok=True)

# Save a sample dataset
sample_data = df_sample.head(200).copy()
data_file = data_dir / 'bike_sample.parquet'
sample_data.to_parquet(data_file, index=False)
file_size = data_file.stat().st_size / 1024

print(f"   ✅ Created: {data_file.name} ({file_size:.1f} KB)")

# Step 5: Track data with DVC
print("\n5️⃣ Tracking data with DVC...")
print("   Command: dvc add data/bike_sample.parquet")

dvc_file = data_dir / 'bike_sample.parquet.dvc'
gitignore_file = data_dir / '.gitignore'

# Show what DVC would create
print(f"\n   DVC creates two files:")
print(f"   • {data_file.name}.dvc    (metadata file, tracked by Git)")
print(f"   • .gitignore              (excludes actual data from Git)")

# Simulate .dvc file content
dvc_content = f"""outs:
- md5: abc123def456  # Hash of actual data file
  size: {int(file_size * 1024)}
  path: bike_sample.parquet
"""

print(f"\n   📄 Content of bike_sample.parquet.dvc:")
print("   " + "\n   ".join(dvc_content.strip().split('\n')))

# Step 6: Set up remote storage
print("\n6️⃣ Configuring remote storage...")
remote_dir = TEST_DIR / 'dvc_remote_storage'
remote_dir.mkdir(exist_ok=True)

print(f"   Local remote path: {remote_dir}")
print(f"   Command: dvc remote add -d myremote {remote_dir}")
print(f"   Command: dvc push  # Uploads data to remote")

# Simulate remote storage
remote_copy = remote_dir / 'ab' / 'c123def456'
remote_copy.parent.mkdir(exist_ok=True, parents=True)
shutil.copy(data_file, remote_copy)
print(f"   ✅ Data copied to remote storage (simulated)")

# Step 7: Create DVC pipeline
print("\n7️⃣ Creating DVC pipeline...")
print("   DVC pipelines define data processing stages")

pipeline_content = """stages:
  prepare_data:
    cmd: python prepare_data.py
    deps:
    - data/raw/bike_data.csv
    - prepare_data.py
    outs:
    - data/processed/bike_clean.parquet
    
  train_model:
    cmd: python train.py
    deps:
    - data/processed/bike_clean.parquet
    - train.py
    outs:
    - models/bike_model.pkl
    metrics:
    - metrics/performance.json
"""

dvc_yaml = dvc_demo_dir / 'dvc.yaml'
with open(dvc_yaml, 'w') as f:
    f.write(pipeline_content)

print(f"   ✅ Created: dvc.yaml (pipeline definition)")
print("\n   Pipeline stages:")
print("   • prepare_data: Raw → Clean data")
print("   • train_model:  Clean data → Model")
print("\n   Run with: dvc repro")

# Summary and workflow
print("\n" + "="*70)
print("📊 DVC WORKFLOW SUMMARY")
print("="*70)

workflow = """
🔄 TYPICAL DVC WORKFLOW:

1. TRACK DATA:
   $ dvc add data/large_dataset.parquet
   $ git add data/large_dataset.parquet.dvc data/.gitignore
   $ git commit -m "Track dataset with DVC"

2. SET UP REMOTE:
   $ dvc remote add -d storage s3://my-bucket/dvc-storage
   $ dvc push  # Upload data to remote
   
3. COLLABORATE:
   Team member:
   $ git pull
   $ dvc pull  # Download data from remote
   
4. CREATE PIPELINE:
   $ dvc stage add -n prepare -d data/raw -o data/processed prepare.py
   $ dvc repro  # Run entire pipeline
   
5. VERSION DATA:
   $ dvc add data/dataset_v2.parquet
   $ git commit -m "Update to v2"
   $ dvc push

🎯 KEY BENEFITS:
   ✓ Git tracks only metadata (KB), not data (GB)
   ✓ Data stored in cloud (S3, GCS, Azure, etc.)
   ✓ Pipeline ensures reproducibility
   ✓ Easy team collaboration
   ✓ Automatic data versioning

💡 WHEN TO USE DVC:
   • Datasets > 100 MB
   • Multiple team members
   • Need reproducible pipelines
   • Cloud storage available
   • Want Git-like data versioning

🎓 LEARNING POINTS:
   ✓ DVC complements Git, doesn't replace it
   ✓ .dvc files are small (track in Git)
   ✓ Actual data goes to remote storage
   ✓ Pipelines track dependencies automatically
   ✓ Works with any cloud provider

📚 PRODUCTION SETUP:
   # AWS S3
   dvc remote add -d storage s3://my-bucket/dvc-cache
   
   # Google Cloud Storage
   dvc remote add -d storage gs://my-bucket/dvc-cache
   
   # Azure Blob
   dvc remote add -d storage azure://my-container/dvc-cache
   
   # SSH/Local
   dvc remote add -d storage ssh://server/path
"""

print(workflow)

print("\n🔗 Resources:")
print("   • DVC Documentation: https://dvc.org/doc")
print("   • DVC Tutorial: https://dvc.org/doc/start")
print("   • DVC with S3: https://dvc.org/doc/user-guide/setup-google-drive-remote")

print("\n✅ Challenge 6.1 complete! DVC setup demonstrated.")
print("="*70)

---

### Challenge 6.2 Solution: Cloud Storage Integration ☁️

In [ ]:
# ═══════════════════════════════════════════════════════════
# CHALLENGE 6.2 SOLUTION: CLOUD STORAGE INTEGRATION
# ═══════════════════════════════════════════════════════════

"""
Cloud storage enables scalable, durable data storage with pay-as-you-go pricing.

This solution demonstrates:
1. Cloud storage abstraction (works with S3, GCS, Azure)
2. Upload/download with caching
3. Error handling and retries
4. Integration with local workflows
"""

import hashlib
from typing import Optional, Dict
from datetime import datetime

print("☁️  CHALLENGE 6.2 SOLUTION: Cloud Storage Integration")
print("="*70)

# Cloud Storage Manager (Mock implementation - works without credentials)
class CloudStorageManager:
    """
    Unified interface for cloud storage providers.
    Supports: AWS S3, Google Cloud Storage, Azure Blob Storage
    """
    
    def __init__(self, provider: str = 'local', cache_dir: Optional[Path] = None):
        """
        Initialize cloud storage manager.
        
        Parameters:
        -----------
        provider : str
            'aws', 'gcs', 'azure', or 'local' (for testing)
        cache_dir : Path
            Local directory for caching downloads
        """
        self.provider = provider
        self.cache_dir = cache_dir or TEST_DIR / 'cloud_cache'
        self.cache_dir.mkdir(exist_ok=True)
        
        # Mock remote storage (simulates cloud)
        self.remote_dir = TEST_DIR / f'{provider}_storage'
        self.remote_dir.mkdir(exist_ok=True)
        
        # Metadata tracking
        self.metadata_file = self.cache_dir / 'cloud_metadata.json'
        self.metadata = self._load_metadata()
        
        print(f"   ✅ Initialized {provider.upper()} storage")
        print(f"   Cache: {self.cache_dir}")
        print(f"   Remote: {self.remote_dir} (simulated)")
    
    def _load_metadata(self) -> Dict:
        """Load cached file metadata"""
        if self.metadata_file.exists():
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return {}
    
    def _save_metadata(self):
        """Save cached file metadata"""
        with open(self.metadata_file, 'w') as f:
            json.dump(self.metadata, f, indent=2)
    
    def _compute_hash(self, filepath: Path) -> str:
        """Compute MD5 hash of file"""
        md5 = hashlib.md5()
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                md5.update(chunk)
        return md5.hexdigest()
    
    def upload(self, local_path: Path, remote_key: str, 
               overwrite: bool = False) -> Dict:
        """
        Upload file to cloud storage.
        
        Parameters:
        -----------
        local_path : Path
            Local file to upload
        remote_key : str
            Remote path/key (e.g., 'datasets/bike_data.parquet')
        overwrite : bool
            Whether to overwrite existing files
        
        Returns:
        --------
        Dict with upload info
        """
        if not local_path.exists():
            raise FileNotFoundError(f"Local file not found: {local_path}")
        
        # Compute hash for integrity checking
        file_hash = self._compute_hash(local_path)
        file_size = local_path.stat().st_size
        
        # Simulate cloud upload (copy to remote storage)
        remote_path = self.remote_dir / remote_key
        remote_path.parent.mkdir(parents=True, exist_ok=True)
        
        if remote_path.exists() and not overwrite:
            print(f"   ⚠️  File exists in cloud: {remote_key}")
            print(f"      Use overwrite=True to replace")
            return {'status': 'skipped', 'reason': 'exists'}
        
        # Simulate upload
        shutil.copy(local_path, remote_path)
        
        # Store metadata
        upload_info = {
            'remote_key': remote_key,
            'file_hash': file_hash,
            'size_bytes': file_size,
            'uploaded_at': datetime.now().isoformat(),
            'provider': self.provider,
            'local_cache': None
        }
        
        self.metadata[remote_key] = upload_info
        self._save_metadata()
        
        print(f"   ✅ Uploaded: {remote_key}")
        print(f"      Size: {file_size/1024:.1f} KB")
        print(f"      Hash: {file_hash[:12]}...")
        
        return upload_info
    
    def download(self, remote_key: str, local_path: Optional[Path] = None,
                 use_cache: bool = True) -> Path:
        """
        Download file from cloud storage with caching.
        
        Parameters:
        -----------
        remote_key : str
            Remote path/key to download
        local_path : Path, optional
            Where to save (defaults to cache)
        use_cache : bool
            Check cache first before downloading
        
        Returns:
        --------
        Path to downloaded file
        """
        # Check cache first
        if use_cache:
            cached_path = self.cache_dir / remote_key.replace('/', '_')
            if cached_path.exists():
                # Verify integrity if metadata exists
                if remote_key in self.metadata:
                    cached_hash = self._compute_hash(cached_path)
                    stored_hash = self.metadata[remote_key]['file_hash']
                    if cached_hash == stored_hash:
                        print(f"   ✅ Using cached: {remote_key}")
                        print(f"      Location: {cached_path}")
                        return cached_path
                    else:
                        print(f"   ⚠️  Cache invalid (hash mismatch), re-downloading")
                else:
                    print(f"   ✅ Using cached: {remote_key}")
                    return cached_path
        
        # Download from cloud
        remote_path = self.remote_dir / remote_key
        if not remote_path.exists():
            raise FileNotFoundError(f"Remote file not found: {remote_key}")
        
        # Determine download location
        if local_path is None:
            local_path = self.cache_dir / remote_key.replace('/', '_')
        
        local_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Simulate download
        shutil.copy(remote_path, local_path)
        file_hash = self._compute_hash(local_path)
        
        # Update metadata
        if remote_key in self.metadata:
            self.metadata[remote_key]['local_cache'] = str(local_path)
            self.metadata[remote_key]['downloaded_at'] = datetime.now().isoformat()
        else:
            self.metadata[remote_key] = {
                'remote_key': remote_key,
                'file_hash': file_hash,
                'local_cache': str(local_path),
                'downloaded_at': datetime.now().isoformat()
            }
        
        self._save_metadata()
        
        print(f"   ✅ Downloaded: {remote_key}")
        print(f"      Saved to: {local_path}")
        
        return local_path
    
    def list_files(self, prefix: str = '') -> list:
        """List files in cloud storage"""
        if prefix:
            pattern = f"{prefix}*"
        else:
            pattern = "**/*"
        
        files = [f.relative_to(self.remote_dir) 
                for f in self.remote_dir.glob(pattern)
                if f.is_file()]
        
        return [str(f) for f in files]
    
    def delete(self, remote_key: str) -> bool:
        """Delete file from cloud storage"""
        remote_path = self.remote_dir / remote_key
        if remote_path.exists():
            remote_path.unlink()
            
            # Remove from metadata
            if remote_key in self.metadata:
                del self.metadata[remote_key]
                self._save_metadata()
            
            print(f"   ✅ Deleted: {remote_key}")
            return True
        return False
    
    def get_stats(self) -> Dict:
        """Get storage statistics"""
        total_size = sum(f.stat().st_size 
                        for f in self.remote_dir.rglob('*') 
                        if f.is_file())
        file_count = len(list(self.remote_dir.rglob('*')))
        
        cache_size = sum(f.stat().st_size 
                        for f in self.cache_dir.rglob('*')
                        if f.is_file())
        cache_count = len(list(self.cache_dir.rglob('*')))
        
        return {
            'remote_files': file_count,
            'remote_size_mb': total_size / 1024 / 1024,
            'cached_files': cache_count,
            'cached_size_mb': cache_size / 1024 / 1024
        }


# Demonstration
print("\n" + "="*70)
print("📊 DEMONSTRATION")
print("="*70)

# Initialize cloud storage manager
cloud = CloudStorageManager(provider='aws', cache_dir=TEST_DIR / 'cloud_cache')

# Step 1: Upload a dataset
print("\n1️⃣ Uploading dataset to cloud...")
sample_file = TEST_DIR / 'bike_sample_for_cloud.parquet'
df_sample.head(300).to_parquet(sample_file, index=False)

upload_result = cloud.upload(
    local_path=sample_file,
    remote_key='datasets/bike_availability/v1/bike_data.parquet'
)

# Step 2: Upload another version
print("\n2️⃣ Uploading second version...")
sample_file_v2 = TEST_DIR / 'bike_sample_v2_for_cloud.parquet'
df_sample.head(400).to_parquet(sample_file_v2, index=False)

cloud.upload(
    local_path=sample_file_v2,
    remote_key='datasets/bike_availability/v2/bike_data.parquet'
)

# Step 3: List files in cloud
print("\n3️⃣ Listing files in cloud storage...")
files = cloud.list_files(prefix='datasets/')
print(f"   Found {len(files)} files:")
for f in files:
    print(f"   • {f}")

# Step 4: Download with caching
print("\n4️⃣ Downloading file (first time - downloads from cloud)...")
downloaded_path = cloud.download('datasets/bike_availability/v1/bike_data.parquet')

print("\n5️⃣ Downloading same file (second time - uses cache)...")
downloaded_path_2 = cloud.download('datasets/bike_availability/v1/bike_data.parquet')

# Step 6: Storage statistics
print("\n6️⃣ Storage statistics...")
stats = cloud.get_stats()
print(f"   Remote storage: {stats['remote_files']} files, {stats['remote_size_mb']:.2f} MB")
print(f"   Local cache:    {stats['cached_files']} files, {stats['cached_size_mb']:.2f} MB")

# Summary and best practices
print("\n" + "="*70)
print("☁️  CLOUD STORAGE BEST PRACTICES")
print("="*70)

best_practices = """
✅ PRODUCTION IMPLEMENTATION:

1. AWS S3 (boto3):
   import boto3
   s3 = boto3.client('s3')
   s3.upload_file('local.parquet', 'my-bucket', 'data/file.parquet')
   s3.download_file('my-bucket', 'data/file.parquet', 'local.parquet')

2. Google Cloud Storage:
   from google.cloud import storage
   client = storage.Client()
   bucket = client.bucket('my-bucket')
   blob = bucket.blob('data/file.parquet')
   blob.upload_from_filename('local.parquet')

3. Azure Blob Storage:
   from azure.storage.blob import BlobServiceClient
   blob_service = BlobServiceClient.from_connection_string(conn_str)
   blob_client = blob_service.get_blob_client('container', 'file.parquet')
   with open('local.parquet', 'rb') as data:
       blob_client.upload_blob(data)

💡 KEY PATTERNS:

   ✓ Always cache downloads locally
   ✓ Use hash verification for integrity
   ✓ Implement retry logic for uploads/downloads
   ✓ Use compression (Parquet already compressed)
   ✓ Organize with meaningful folder structure
   ✓ Tag/label files with metadata
   ✓ Set up lifecycle policies (auto-delete old files)
   ✓ Enable versioning on bucket/container
   ✓ Use IAM roles, not hardcoded credentials

🎯 WHEN TO USE CLOUD STORAGE:

   • Datasets too large for local disk
   • Need to share data across team
   • Want automatic backups and durability
   • Need to scale storage dynamically
   • Working with distributed compute (Spark, Dask)

💰 COST OPTIMIZATION:

   • Use appropriate storage class (Standard, Infrequent Access, Archive)
   • Compress data before uploading
   • Delete old/unused files
   • Use lifecycle policies for automatic archival
   • Monitor data transfer costs (egress)

🔒 SECURITY:

   • Never commit credentials to Git
   • Use environment variables or secrets manager
   • Enable encryption at rest and in transit
   • Use signed URLs for temporary access
   • Implement least-privilege access policies

📊 INTEGRATION WITH PANDAS:

   # Direct reading from cloud (requires credentials)
   import pandas as pd
   
   # S3
   df = pd.read_parquet('s3://bucket/data.parquet')
   
   # GCS
   df = pd.read_parquet('gs://bucket/data.parquet')
   
   # Azure
   df = pd.read_parquet('az://container/data.parquet')
"""

print(best_practices)

print("\n🔗 Provider-Specific Resources:")
print("   • AWS S3: https://boto3.amazonaws.com/v1/documentation/api/latest/guide/s3.html")
print("   • Google Cloud: https://cloud.google.com/storage/docs/reference/libraries")
print("   • Azure Blob: https://docs.microsoft.com/en-us/azure/storage/blobs/")

print("\n✅ Challenge 6.2 complete! Cloud storage patterns demonstrated.")
print("="*70)

---

### Challenge 6.3 Solution: Streaming Data Storage 📡

In [ ]:
# ═══════════════════════════════════════════════════════════
# CHALLENGE 6.3 SOLUTION: STREAMING DATA STORAGE
# ═══════════════════════════════════════════════════════════

"""
Streaming data storage handles continuous data ingestion with:
- Time-based partitioning for efficient queries
- Append-only writes for consistency
- Late data handling
- Automatic compaction

This solution demonstrates a production-grade streaming storage system.
"""

from typing import List, Tuple
import pyarrow as pa
import pyarrow.parquet as pq

print("📡 CHALLENGE 6.3 SOLUTION: Streaming Data Storage")
print("="*70)

class StreamingStorageManager:
    """
    Time-partitioned storage for streaming bike availability data.
    
    Features:
    - Hourly partitions for efficient queries
    - Append-only writes
    - Late data handling
    - Query interface for date ranges
    - Automatic compaction
    """
    
    def __init__(self, base_dir: Path):
        self.base_dir = base_dir
        self.base_dir.mkdir(exist_ok=True)
        
        # Partition structure: year/month/day/hour/
        self.partition_format = "{year:04d}/{month:02d}/{day:02d}/{hour:02d}"
        
        print(f"✅ Initialized streaming storage")
        print(f"   Base: {base_dir}")
        print(f"   Partitioning: Year/Month/Day/Hour")
    
    def _get_partition_path(self, timestamp: pd.Timestamp) -> Path:
        """Get partition directory for a timestamp"""
        partition_str = self.partition_format.format(
            year=timestamp.year,
            month=timestamp.month,
            day=timestamp.day,
            hour=timestamp.hour
        )
        return self.base_dir / partition_str
    
    def append_data(self, df: pd.DataFrame, timestamp_col: str = 'timestamp'):
        """
        Append streaming data with time-based partitioning.
        
        Parameters:
        -----------
        df : DataFrame
            Data to append (must have timestamp column)
        timestamp_col : str
            Name of timestamp column for partitioning
        """
        if timestamp_col not in df.columns:
            raise ValueError(f"Timestamp column '{timestamp_col}' not found")
        
        # Ensure timestamp is datetime
        df[timestamp_col] = pd.to_datetime(df[timestamp_col])
        
        # Group by hour for partitioning
        df['_partition_hour'] = df[timestamp_col].dt.floor('H')
        
        records_written = 0
        partitions_written = set()
        
        for partition_hour, partition_data in df.groupby('_partition_hour'):
            # Get partition path
            partition_path = self._get_partition_path(partition_hour)
            partition_path.mkdir(parents=True, exist_ok=True)
            
            # Create filename with microsecond precision for uniqueness
            filename = f"data_{partition_hour.strftime('%Y%m%d_%H%M%S')}_{datetime.now().strftime('%f')}.parquet"
            file_path = partition_path / filename
            
            # Remove temporary partition column
            partition_data = partition_data.drop(columns=['_partition_hour'])
            
            # Write partition
            partition_data.to_parquet(file_path, index=False, compression='snappy')
            
            records_written += len(partition_data)
            partitions_written.add(str(partition_path.relative_to(self.base_dir)))
        
        print(f"   ✅ Appended {records_written} records to {len(partitions_written)} partitions")
        return {
            'records': records_written,
            'partitions': list(partitions_written)
        }
    
    def query_range(self, start_time: pd.Timestamp, end_time: pd.Timestamp) -> pd.DataFrame:
        """
        Query data for a time range efficiently using partitions.
        
        Parameters:
        -----------
        start_time : Timestamp
            Start of time range (inclusive)
        end_time : Timestamp
            End of time range (inclusive)
        
        Returns:
        --------
        DataFrame with data in time range
        """
        # Generate all hours in range
        hours = pd.date_range(start=start_time.floor('H'), 
                             end=end_time.ceil('H'), 
                             freq='H')
        
        dfs = []
        partitions_read = 0
        
        for hour in hours:
            partition_path = self._get_partition_path(hour)
            
            if not partition_path.exists():
                continue
            
            # Read all files in partition
            for file_path in partition_path.glob('*.parquet'):
                df_partition = pd.read_parquet(file_path)
                dfs.append(df_partition)
                partitions_read += 1
        
        if not dfs:
            print(f"   ℹ️  No data found for range {start_time} to {end_time}")
            return pd.DataFrame()
        
        # Combine and filter to exact time range
        df_result = pd.concat(dfs, ignore_index=True)
        df_result['timestamp'] = pd.to_datetime(df_result['timestamp'])
        df_result = df_result[
            (df_result['timestamp'] >= start_time) & 
            (df_result['timestamp'] <= end_time)
        ].sort_values('timestamp')
        
        print(f"   ✅ Read {len(df_result)} records from {partitions_read} partition files")
        
        return df_result
    
    def get_latest(self, hours: int = 24) -> pd.DataFrame:
        """Get most recent data"""
        end_time = pd.Timestamp.now()
        start_time = end_time - pd.Timedelta(hours=hours)
        return self.query_range(start_time, end_time)
    
    def compact_partitions(self, before_date: pd.Timestamp = None):
        """
        Compact small partition files into larger ones for efficiency.
        
        Parameters:
        -----------
        before_date : Timestamp
            Compact partitions before this date (default: yesterday)
        """
        if before_date is None:
            before_date = pd.Timestamp.now() - pd.Timedelta(days=1)
        
        print(f"   🔄 Compacting partitions before {before_date.date()}...")
        
        compacted_count = 0
        
        # Find all partition directories
        for year_dir in self.base_dir.glob('[0-9][0-9][0-9][0-9]'):
            for month_dir in year_dir.glob('[0-9][0-9]'):
                for day_dir in month_dir.glob('[0-9][0-9]'):
                    for hour_dir in day_dir.glob('[0-9][0-9]'):
                        # Parse partition timestamp
                        parts = hour_dir.relative_to(self.base_dir).parts
                        partition_time = pd.Timestamp(
                            year=int(parts[0]),
                            month=int(parts[1]),
                            day=int(parts[2]),
                            hour=int(parts[3])
                        )
                        
                        if partition_time >= before_date:
                            continue
                        
                        # Get all files in partition
                        files = list(hour_dir.glob('*.parquet'))
                        
                        if len(files) <= 1:
                            continue  # Already compacted
                        
                        # Read all files
                        dfs = [pd.read_parquet(f) for f in files]
                        df_combined = pd.concat(dfs, ignore_index=True)
                        
                        # Write compacted file
                        compacted_file = hour_dir / f"compacted_{partition_time.strftime('%Y%m%d_%H')}.parquet"
                        df_combined.to_parquet(compacted_file, index=False, compression='snappy')
                        
                        # Delete original files
                        for f in files:
                            f.unlink()
                        
                        compacted_count += 1
        
        print(f"   ✅ Compacted {compacted_count} partitions")
        return compacted_count
    
    def get_storage_stats(self) -> Dict:
        """Get storage statistics"""
        all_files = list(self.base_dir.rglob('*.parquet'))
        
        total_size = sum(f.stat().st_size for f in all_files)
        
        # Count partitions (directories with files)
        partitions = set(f.parent for f in all_files)
        
        return {
            'total_files': len(all_files),
            'total_partitions': len(partitions),
            'total_size_mb': total_size / 1024 / 1024,
            'avg_file_size_kb': (total_size / len(all_files) / 1024) if all_files else 0
        }


# Demonstration
print("\n" + "="*70)
print("📊 DEMONSTRATION: Streaming Data Storage")
print("="*70)

# Initialize streaming storage
streaming_dir = TEST_DIR / 'streaming_storage'
stream = StreamingStorageManager(streaming_dir)

# Simulate streaming data ingestion
print("\n1️⃣ Simulating streaming data ingestion...")

# Generate 3 days of hourly data (72 hours)
base_time = pd.Timestamp('2026-01-13 00:00:00')
streaming_data = []

for hour_offset in range(72):
    timestamp = base_time + pd.Timedelta(hours=hour_offset)
    
    # Simulate 5 stations reporting every hour
    for station_id in range(5):
        streaming_data.append({
            'timestamp': timestamp,
            'station_id': f'Station_{station_id:03d}',
            'bikes_available': np.random.randint(0, 25),
            'docks_available': np.random.randint(0, 10),
            'temperature_c': 10 + np.random.uniform(-3, 5)
        })

df_stream = pd.DataFrame(streaming_data)
print(f"   Generated {len(df_stream)} records over {72} hours")

# Append data in batches (simulating hourly ingestion)
print("\n2️⃣ Ingesting data in batches...")
batch_size = 50  # Records per batch

for i in range(0, len(df_stream), batch_size):
    batch = df_stream.iloc[i:i+batch_size]
    result = stream.append_data(batch)
    if i == 0 or (i // batch_size) % 10 == 0:
        print(f"   Batch {i//batch_size + 1}: {result['records']} records → {len(result['partitions'])} partitions")

# Query examples
print("\n3️⃣ Querying data...")

# Query last 24 hours
query_start = base_time + pd.Timedelta(days=2)
query_end = base_time + pd.Timedelta(days=3)

print(f"\n   Query: {query_start} to {query_end}")
df_query = stream.query_range(query_start, query_end)
print(f"   Result: {len(df_query)} records")

# Query specific day
print(f"\n   Query: Single day ({base_time.date()})")
day_start = base_time
day_end = base_time + pd.Timedelta(days=1)
df_day = stream.query_range(day_start, day_end)
print(f"   Result: {len(df_day)} records")

# Storage statistics
print("\n4️⃣ Storage statistics before compaction...")
stats_before = stream.get_storage_stats()
print(f"   Files: {stats_before['total_files']}")
print(f"   Partitions: {stats_before['total_partitions']}")
print(f"   Total size: {stats_before['total_size_mb']:.2f} MB")
print(f"   Avg file size: {stats_before['avg_file_size_kb']:.1f} KB")

# Compact old partitions
print("\n5️⃣ Compacting old partitions...")
compact_before = base_time + pd.Timedelta(days=2)
stream.compact_partitions(before_date=compact_before)

stats_after = stream.get_storage_stats()
print(f"\n   After compaction:")
print(f"   Files: {stats_after['total_files']} (reduced by {stats_before['total_files'] - stats_after['total_files']})")

# Summary and patterns
print("\n" + "="*70)
print("📡 STREAMING STORAGE PATTERNS")
print("="*70)

patterns = """
✅ KEY DESIGN DECISIONS:

1. PARTITIONING STRATEGY:
   • Hourly: Real-time dashboards, frequent queries
   • Daily: Batch processing, historical analysis
   • Monthly: Long-term archival, infrequent access
   
   Choose based on query patterns!

2. FILE SIZE OPTIMIZATION:
   • Target: 128-512 MB per file
   • Too small: Metadata overhead, slow queries
   • Too large: Cannot skip irrelevant data
   • Solution: Compact periodically

3. LATE DATA HANDLING:
   • Accept late arrivals up to N hours
   • Write to appropriate partition (not latest)
   • Recompact affected partitions
   • Set SLA for data freshness

4. QUERY OPTIMIZATION:
   • Partition pruning: Only read relevant partitions
   • Predicate pushdown: Filter at read time
   • Columnar format: Read only needed columns
   • Metadata caching: Avoid repeated directory scans

💡 PRODUCTION PATTERNS:

   # Apache Parquet Partitioning
   df.to_parquet('data/streaming', 
                 partition_cols=['year', 'month', 'day', 'hour'],
                 compression='snappy')
   
   # Apache Arrow for Efficient Reading
   import pyarrow.parquet as pq
   dataset = pq.ParquetDataset('data/streaming', 
                                filters=[('year', '=', 2026),
                                        ('month', '=', 1)])
   
   # Dask for Large-Scale Processing
   import dask.dataframe as dd
   ddf = dd.read_parquet('data/streaming')
   result = ddf[ddf.timestamp > '2026-01-01'].compute()

🔄 COMPACTION STRATEGIES:

   1. Time-based: Compact partitions older than X days
   2. Size-based: Compact when partition has >N small files
   3. Scheduled: Run compaction nightly
   4. Triggered: Compact after ingestion burst

⚡ REAL-TIME CONSIDERATIONS:

   • Write latency: < 1 second
   • Query latency: < 5 seconds for recent data
   • Consistency: Append-only ensures read consistency
   • Scalability: Horizontal scaling via partitioning

🎯 USE CASES:

   ✓ IoT sensor data streams
   ✓ Financial tick data
   ✓ Application logs
   ✓ User activity events
   ✓ Real-time bike/vehicle tracking

🛠️ PRODUCTION TOOLS:

   • Apache Kafka: Streaming ingestion
   • Apache Spark Structured Streaming: Processing
   • Delta Lake: ACID transactions + time travel
   • Apache Iceberg: Table format for analytics
   • Apache Hudi: Incremental processing
"""

print(patterns)

print("\n🔗 Further Learning:")
print("   • Delta Lake: https://delta.io/")
print("   • Apache Iceberg: https://iceberg.apache.org/")
print("   • Streaming with Spark: https://spark.apache.org/streaming/")

print("\n✅ Challenge 6.3 complete! Streaming storage patterns demonstrated.")
print("="*70)

---

### Challenge 6.4 Solution: Data Lineage Tracker 📊

In [ ]:
# ═══════════════════════════════════════════════════════════
# CHALLENGE 6.4 SOLUTION: DATA LINEAGE TRACKER
# ═══════════════════════════════════════════════════════════

"""
Data lineage tracks the complete lifecycle of data:
- Where data came from (inputs)
- What transformations were applied
- Where data went (outputs)
- Who/when/why for audit trail

Critical for: data governance, debugging, compliance, reproducibility
"""

from typing import List, Dict, Optional
from dataclasses import dataclass, field, asdict
from enum import Enum

print("📊 CHALLENGE 6.4 SOLUTION: Data Lineage Tracker")
print("="*70)

class TransformationType(Enum):
    """Types of data transformations"""
    FILTER = "filter"
    AGGREGATE = "aggregate"
    JOIN = "join"
    CLEAN = "clean"
    ENRICH = "enrich"
    DERIVE = "derive"
    VALIDATE = "validate"

@dataclass
class DataAsset:
    """Represents a data asset (input or output)"""
    path: str
    name: str
    format: str
    size_bytes: Optional[int] = None
    rows: Optional[int] = None
    columns: Optional[List[str]] = None
    hash: Optional[str] = None
    
    def to_dict(self):
        return asdict(self)

@dataclass
class Transformation:
    """Represents a data transformation"""
    name: str
    type: TransformationType
    description: str
    parameters: Dict = field(default_factory=dict)
    code_snippet: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def to_dict(self):
        d = asdict(self)
        d['type'] = self.type.value
        return d

class DataLineage:
    """
    Tracks complete data lineage for a workflow.
    
    Features:
    - Record inputs, transformations, outputs
    - Generate lineage graphs
    - Export as JSON
    - Query lineage history
    """
    
    def __init__(self, workflow_name: str, workflow_id: Optional[str] = None):
        self.workflow_name = workflow_name
        self.workflow_id = workflow_id or datetime.now().strftime('%Y%m%d_%H%M%S')
        
        self.inputs: List[DataAsset] = []
        self.transformations: List[Transformation] = []
        self.outputs: List[DataAsset] = []
        
        self.metadata = {
            'workflow_name': workflow_name,
            'workflow_id': self.workflow_id,
            'created_at': datetime.now().isoformat(),
            'created_by': 'data_scientist',  # Could be os.getenv('USER')
            'environment': 'development'
        }
        
        print(f"📋 Initialized lineage tracker")
        print(f"   Workflow: {workflow_name}")
        print(f"   ID: {self.workflow_id}")
    
    def add_input(self, path: str, name: str, format: str = 'parquet',
                  df: Optional[pd.DataFrame] = None, **kwargs) -> DataAsset:
        """Add input dataset"""
        asset = DataAsset(
            path=path,
            name=name,
            format=format,
            **kwargs
        )
        
        # Extract metadata from DataFrame if provided
        if df is not None:
            asset.rows = len(df)
            asset.columns = list(df.columns)
            asset.size_bytes = df.memory_usage(deep=True).sum()
        
        self.inputs.append(asset)
        print(f"   ➕ Input: {name} ({format})")
        return asset
    
    def add_transformation(self, name: str, type: TransformationType,
                          description: str, parameters: Dict = None,
                          code: str = None) -> Transformation:
        """Add transformation step"""
        transform = Transformation(
            name=name,
            type=type,
            description=description,
            parameters=parameters or {},
            code_snippet=code
        )
        
        self.transformations.append(transform)
        print(f"   🔄 Transform: {name} ({type.value})")
        return transform
    
    def add_output(self, path: str, name: str, format: str = 'parquet',
                   df: Optional[pd.DataFrame] = None, **kwargs) -> DataAsset:
        """Add output dataset"""
        asset = DataAsset(
            path=path,
            name=name,
            format=format,
            **kwargs
        )
        
        # Extract metadata from DataFrame if provided
        if df is not None:
            asset.rows = len(df)
            asset.columns = list(df.columns)
            asset.size_bytes = df.memory_usage(deep=True).sum()
        
        self.outputs.append(asset)
        print(f"   ✅ Output: {name} ({format})")
        return asset
    
    def to_dict(self) -> Dict:
        """Export complete lineage as dictionary"""
        return {
            'metadata': self.metadata,
            'inputs': [inp.to_dict() for inp in self.inputs],
            'transformations': [t.to_dict() for t in self.transformations],
            'outputs': [out.to_dict() for out in self.outputs]
        }
    
    def to_json(self, filepath: Path) -> Path:
        """Save lineage to JSON file"""
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)
        
        print(f"   💾 Saved lineage: {filepath.name}")
        return filepath
    
    def visualize_text(self):
        """Create text-based visualization"""
        print("\n" + "="*70)
        print(f"📊 DATA LINEAGE: {self.workflow_name}")
        print("="*70)
        
        # Inputs
        print(f"\n📥 INPUTS ({len(self.inputs)}):")
        for inp in self.inputs:
            rows_str = f"{inp.rows:,} rows" if inp.rows else "unknown rows"
            cols_str = f"{len(inp.columns)} cols" if inp.columns else ""
            print(f"   • {inp.name} ({inp.format}) - {rows_str} {cols_str}")
        
        # Transformations
        print(f"\n🔄 TRANSFORMATIONS ({len(self.transformations)}):")
        for i, trans in enumerate(self.transformations, 1):
            print(f"   {i}. {trans.name} [{trans.type.value}]")
            print(f"      {trans.description}")
            if trans.parameters:
                print(f"      Params: {trans.parameters}")
        
        # Outputs
        print(f"\n📤 OUTPUTS ({len(self.outputs)}):")
        for out in self.outputs:
            rows_str = f"{out.rows:,} rows" if out.rows else "unknown rows"
            size_str = f"{out.size_bytes/1024:.1f} KB" if out.size_bytes else ""
            print(f"   • {out.name} ({out.format}) - {rows_str} {size_str}")
        
        print("\n" + "="*70)
    
    def visualize_graph(self, output_path: Optional[Path] = None):
        """
        Create visual graph using ASCII art.
        (In production, use graphviz or networkx for real visualizations)
        """
        print("\n📈 LINEAGE GRAPH:")
        print("="*70)
        
        # Draw inputs
        for inp in self.inputs:
            print(f"📥 {inp.name}")
            print("  │")
        
        # Draw transformations
        for i, trans in enumerate(self.transformations):
            print(f"  ↓")
            print(f"🔄 {trans.name}")
            if i < len(self.transformations) - 1:
                print("  │")
        
        # Draw outputs
        if self.transformations:
            print("  ↓")
        for out in self.outputs:
            print(f"📤 {out.name}")
        
        print("="*70)
        
        print("\n💡 For production graphs, use:")
        print("   • graphviz: pip install graphviz")
        print("   • networkx: pip install networkx matplotlib")
        print("   • Apache Atlas: Enterprise data governance")


# Demonstration
print("\n" + "="*70)
print("📊 DEMONSTRATION: Data Lineage Tracking")
print("="*70)

# Create lineage tracker for bike data workflow
print("\n1️⃣ Initializing lineage tracker...")
lineage = DataLineage(
    workflow_name="bike_availability_etl",
    workflow_id="demo_20260116"
)

# Track inputs
print("\n2️⃣ Recording input datasets...")
lineage.add_input(
    path='data/raw/bike_api_data.csv',
    name='bike_raw',
    format='csv',
    df=df_sample  # Extracts metadata from DataFrame
)

lineage.add_input(
    path='data/raw/weather_api_data.csv',
    name='weather_raw',
    format='csv'
)

# Track transformations
print("\n3️⃣ Recording transformations...")

lineage.add_transformation(
    name='remove_nulls',
    type=TransformationType.CLEAN,
    description='Remove rows with missing critical values',
    parameters={'threshold': 0.1, 'columns': ['bikes_available', 'timestamp']},
    code='df.dropna(subset=["bikes_available", "timestamp"])'
)

lineage.add_transformation(
    name='merge_weather',
    type=TransformationType.JOIN,
    description='Join bike data with weather data on timestamp',
    parameters={'join_key': 'timestamp', 'join_type': 'left'},
    code='pd.merge(bikes, weather, on="timestamp", how="left")'
)

lineage.add_transformation(
    name='add_bikeability_score',
    type=TransformationType.DERIVE,
    description='Calculate bikeability score based on weather',
    parameters={'formula': 'weighted combination of temp, precip, wind'},
    code='df["bikeability"] = calculate_score(df)'
)

lineage.add_transformation(
    name='validate_output',
    type=TransformationType.VALIDATE,
    description='Ensure data quality requirements met',
    parameters={'checks': ['no_nulls', 'date_range', 'value_ranges']}
)

# Track outputs
print("\n4️⃣ Recording outputs...")
df_processed = df_sample.head(200).copy()
df_processed['bikeability_score'] = np.random.uniform(0.5, 1.0, len(df_processed))

lineage.add_output(
    path='data/processed/bike_weather_enriched.parquet',
    name='bike_weather_enriched',
    format='parquet',
    df=df_processed
)

lineage.add_output(
    path='data/processed/bike_weather_enriched_v2.parquet',
    name='bike_weather_v2',
    format='parquet'
)

# Visualize lineage
print("\n5️⃣ Visualizing lineage...")
lineage.visualize_text()
lineage.visualize_graph()

# Export to JSON
print("\n6️⃣ Exporting lineage...")
lineage_file = TEST_DIR / f'lineage_{lineage.workflow_id}.json'
lineage.to_json(lineage_file)

print(f"\n📄 Lineage JSON preview:")
lineage_dict = lineage.to_dict()
print(json.dumps(lineage_dict, indent=2)[:500] + "...")

# Integration example
print("\n" + "="*70)
print("🔗 INTEGRATION WITH SAVE WORKFLOW")
print("="*70)

integration_example = """
# Enhanced DataVersionManager with Lineage
class DataVersionManagerWithLineage(DataVersionManager):
    def save_version(self, df, name, description, lineage=None):
        # Save data
        version_info = super().save_version(df, name, description)
        
        # Save lineage alongside data
        if lineage:
            lineage_path = self.base_dir / f'{name}_lineage_v{version_info["version"]}.json'
            lineage.to_json(lineage_path)
            version_info['lineage_file'] = str(lineage_path)
        
        return version_info

# Usage
vm = DataVersionManagerWithLineage(Path('data/versioned'))

lineage = DataLineage('daily_update', 'v20260116')
lineage.add_input('raw_data.csv', 'raw', 'csv')
lineage.add_transformation('clean', TransformationType.CLEAN, 'Remove nulls')
lineage.add_output('clean_data.parquet', 'clean', 'parquet')

# Save with lineage
vm.save_version(df_processed, 'bike_data', 'Daily update', lineage=lineage)
"""

print(integration_example)

# Summary
print("\n" + "="*70)
print("📊 DATA LINEAGE BEST PRACTICES")
print("="*70)

best_practices = """
✅ WHAT TO TRACK:

   1. Inputs:
      • Source system/API
      • File path and format
      • Schema and statistics
      • Data quality metrics
      
   2. Transformations:
      • Operation type (filter, join, aggregate)
      • Parameters and thresholds
      • Code/SQL executed
      • Performance metrics
      
   3. Outputs:
      • Destination and format
      • Schema changes
      • Row counts and size
      • Validation results

💡 KEY BENEFITS:

   ✓ Debugging: Trace issues back to source
   ✓ Impact Analysis: What's affected by changes?
   ✓ Compliance: Audit trail for regulations
   ✓ Reproducibility: Recreate results exactly
   ✓ Documentation: Self-documenting workflows

🎯 PRODUCTION TOOLS:

   • Apache Atlas: Enterprise data governance
   • OpenLineage: Open standard for lineage
   • Great Expectations: Data validation with lineage
   • dbt: Analytics engineering with lineage graph
   • Airflow: Workflow orchestration with lineage
   • Amundsen: Data discovery with lineage

🔒 GOVERNANCE USE CASES:

   • GDPR Compliance: Track personal data flow
   • Data Quality: Identify upstream issues
   • Change Impact: See downstream dependencies
   • Cost Attribution: Track data usage
   • Security: Monitor sensitive data access

🛠️ IMPLEMENTATION PATTERNS:

   1. Decorator Pattern:
      @track_lineage
      def transform_data(df):
          return df.dropna()
   
   2. Context Manager:
      with LineageTracker('workflow') as lineage:
          df = process_data()
          lineage.track(df)
   
   3. Framework Integration:
      # Apache Spark
      spark.conf.set("spark.sql.queryExecutionListeners", "LineageListener")
      
      # dbt
      # Automatic lineage from SQL dependencies

📚 STANDARDS:

   • OpenLineage: https://openlineage.io/
   • DCAT (Data Catalog Vocabulary): W3C standard
   • PROV-O (Provenance Ontology): W3C standard
"""

print(best_practices)

print("\n🔗 Resources:")
print("   • OpenLineage: https://openlineage.io/")
print("   • Apache Atlas: https://atlas.apache.org/")
print("   • Great Expectations: https://greatexpectations.io/")
print("   • dbt: https://www.getdbt.com/")

print("\n✅ Challenge 6.4 complete! Data lineage tracking demonstrated.")
print("="*70)

---

## 📝 Part 10: Summary

### What You've Learned ✅

In this notebook, you:
1. ✅ Compared file formats (CSV, JSON, Parquet, Feather)
2. ✅ Measured storage size and I/O performance
3. ✅ Implemented data versioning with metadata
4. ✅ Created comprehensive data documentation
5. ✅ Set up .gitignore best practices
6. ✅ Built a data catalog system
7. ✅ Learned complete save workflow with all best practices

### Key Takeaways 💡

1. **Format matters** - Parquet is best for processed data
2. **Version everything** - Track changes over time
3. **Document thoroughly** - Future you will thank current you
4. **Never commit large files** - Use .gitignore properly
5. **Catalog your data** - Make datasets discoverable
6. **Automate workflows** - Create reusable save functions

### Best Practices Checklist ✅

- [ ] Store raw data in original format (immutable)
- [ ] Use Parquet for processed data (compressed, fast)
- [ ] Version datasets with clear naming
- [ ] Create metadata for every dataset
- [ ] Update .gitignore to exclude large files
- [ ] Maintain a data catalog
- [ ] Document transformations in README files
- [ ] Back up important data

### Next Steps 🚀

Now that you understand storage patterns, proceed to:
- **M2_04_merge_datasets.ipynb** - Combine bike and weather data with proper storage

### 🧠 Reflection Questions

1. **When would you choose CSV** over Parquet despite its larger size?
2. **How would you handle** versioning for a dataset that updates daily?
3. **What information** should always be in data documentation?
4. **How would you organize** data for a team of 10 data scientists?

**Write your reflections below** ⬇️

### My Reflections

[Your thoughts here]

---

## 📚 References

- [Pandas I/O Tools](https://pandas.pydata.org/docs/user_guide/io.html)
- [Parquet Format Documentation](https://parquet.apache.org/docs/)
- [Data Version Control (DVC)](https://dvc.org/)
- [Git LFS Documentation](https://git-lfs.github.com/)
- [Data Management Best Practices](https://the-turing-way.netlify.app/reproducible-research/rdm.html)

---

**🎉 Congratulations!** You've successfully completed M2_03 - Data Storage Patterns!